# UBS and JPMorgan earnings calls, 2022 to 2025: topics and themes around the two mergers

Willian Angelo · Team 10 · Bank of England employer project (CAM_DS_401) · release `2026-09-30`

The topic track for Assignment 3. The release of 30 September 2026 covers both banks from 2022-Q1 to 2025-Q4, so the conversation can be read a year before the 2023 deals and two years after them. The analyst and management Q&A is modelled two ways. A Sentence-BERT tagger, fine-tuned here on 56 example sentences, assigns each sentence to one of seven fixed themes. BERTopic finds the topics without being told what to look for, and its setting is chosen by published measures of stability, coherence and diversity. The results are then set against the quarters the team's sentiment models flagged, and against their labels sentence by sentence (topics within negative sentences). Everything runs on this release only; `angelo.ipynb` stays as the Assignment 2 record.

Outputs print transcript text. **Clear all outputs before every commit** (public repository).

## Environment

Team setup cell with `NAME = "angelo"`, then `EXPORT = "2026-09-30"`. The cells below expect `ROOT`, `DATA`, `EXPORT` and `loading`.

In [ ]:
# 0.0 local Windows only, run before the setup cell: pandas loads pyarrow, whose bundled msvcp140.dll
# is older than the one torch needs, and torch then fails with WinError 1114. Harmless in Colab.
import torch

Loads PyTorch before anything else. On Windows, loading pandas first stops PyTorch from starting, so this line only keeps the notebook running; it does no analysis.

In [ ]:
# project setup - run first, do not edit except NAME
NAME = "angelo"  # <<< your name

import sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules
REPO = "boe-financial-analysis"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    if not pathlib.Path(f"/content/{REPO}").exists():
        subprocess.run(["git", "clone", "-q",
                        f"https://github.com/maybebool/{REPO}.git",
                        f"/content/{REPO}"], check=True)
    ROOT = pathlib.Path(f"/content/{REPO}")
    DATA = pathlib.Path("/content/drive/MyDrive/boe-data")
else:
    ROOT = pathlib.Path.cwd()
    while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    DATA = ROOT / "data"

sys.path.insert(0, str(ROOT))
subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin"], check=False)
subprocess.run(["git", "-C", str(ROOT), "merge", "-q", "origin/main", "-m", "sync"],
               check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(ROOT / "requirements.txt")], check=True)

from notebooks.env_cell import verify, check_python
check_python()
try:
    verify(ROOT)
except RuntimeError as e:
    print(e)
    print("\n Runtime -> Restart session, then run this cell again.")
    raise

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src import loading

print(f"ok  python {sys.version.split()[0]}  pandas {pd.__version__}  data {DATA}")

Standard project set-up: finds the project folder, updates the code from the team's shared branch, installs the pinned library versions and checks them. The output confirms Python 3.13 and pandas 2.2.3, so the results can be reproduced on another machine.

In [ ]:
EXPORT = "2026-09-30"  # pin the export this notebook was written against
print("exports:", loading.exports(DATA))
print("files:  ", loading.files(DATA, EXPORT))

sentences_df = loading.load(DATA, "structured_sentences.csv", EXPORT)

Fixes the analysis to the 30 September release, which extends the data to 2022 to 2025, and loads its sentence file: 19,092 sentences from 33 calls. The release has four files: sentences, speaker turns, Q&A exchanges and the exchange questions.

In [ ]:
sentences_df

A first look at the data: one row per sentence, with the bank, quarter, call type, section (prepared remarks or Q&A), speaker, the sentence text and a flag for filler sentences such as "Thank you.".

## Data import and cleaning

In [ ]:
# 1.0 settings shared by every section
try:
    import torch
except OSError as e:
    raise OSError("torch failed to load because pandas loaded first: restart the kernel "
                  "and run the 0.0 cell before the setup cell") from e
import gc, re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

for name in ("ROOT", "DATA", "EXPORT", "loading"):
    if name not in globals():
        raise NameError(f"run the Environment cell first: {name} is not defined")
if EXPORT not in loading.exports(DATA):
    raise FileNotFoundError(f"release {EXPORT} is not in {DATA / 'results'}")

SEED = 42
OUT = ROOT / "notebooks" / "angelo" / "outputs" / EXPORT      # outputs/ is kept out of git by .git/info/exclude
if DATA in OUT.parents:
    raise RuntimeError(f"never write inside {DATA}: it holds the read-only release")
OUT.mkdir(parents=True, exist_ok=True)
FIG = OUT / "figures"                                           # charts saved for the slides
FIG.mkdir(exist_ok=True)

BANKS = ["UBS", "JPM"]
ROLES = ["analyst", "management"]
YEARS = [2022, 2023, 2024, 2025]
QUARTERS = [f"{y}-Q{q}" for y in YEARS for q in (1, 2, 3, 4)]
CALL_ID = ["bank", "quarter", "call_type"]
TURN = CALL_ID + ["position_in_call"]
KEY = TURN + ["sentence_number"]               # joins results across releases and across the team's tracks

# windows around the 2023 deals, the same for both banks (proposed): the year before, the two deal quarters
# (UBS announced Credit Suisse in March and closed in June; JPMorgan bought First Republic on 1 May),
# the first year after, and the six later quarters
WINDOWS = {"before": QUARTERS[0:4], "deal": QUARTERS[4:6], "after": QUARTERS[6:10], "later": QUARTERS[10:16]}
WINDOW_OF = {q: w for w, qs in WINDOWS.items() for q in qs}
WINDOW_NAMES = list(WINDOWS)

# breaks marked on every time chart
BREAKS = {"UBS": ("2023-Q2", "Credit Suisse consolidated"),
          "JPM": ("2024-Q2", "Visa gain, CFO-only call")}

# the one event call: JPMorgan's First Republic deal call (1 May 2023), filed under 2023-Q2 next to that
# quarter's earnings call. It stays in the corpus; results per quarter pool it with the earnings call.
EVENT = ("JPM", "2023-Q2", "event")
EVENT_NOTE = "Diamond: JPM's First Republic deal call on its own (1 May 2023)"

# chart style: one fixed colour per bank, recessive grid and axes
BANK_COLOR = {"UBS": "#2a78d6", "JPM": "#eb6834"}
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7", "axes.grid": True, "axes.axisbelow": True,
    "grid.color": "#e1e0d9", "grid.linewidth": 0.6, "lines.linewidth": 2,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.labelcolor": "#52514e",
    "xtick.color": "#52514e", "ytick.color": "#52514e", "legend.frameon": False,
})
pd.set_option("display.max_colwidth", 120)
print(f"release {EXPORT}   outputs {OUT}")

Sets the constants used throughout: the two banks, the 16 quarters from 2022 to 2025, a random seed for repeatable results and the output folder. It also defines four windows around the 2023 acquisitions (UBS bought Credit Suisse, JPMorgan bought First Republic): the year before, the two deal quarters, the first year after and the six later quarters. Two quarters with known breaks in the reported figures and JPMorgan's deal call are marked so the charts can show them.

In [ ]:
# 1.1 load the pinned release: sentences, speaker turns (utterances) and the Q&A exchanges
sent_raw = loading.load(DATA, "structured_sentences.csv", EXPORT)
utt_raw = loading.load(DATA, "structured_utterances.csv", EXPORT)
exch_raw = loading.load(DATA, "qa_exchanges_utterance_level.csv", EXPORT)

Loads the three tables used: 19,092 sentences, 2,570 speaker turns and 2,473 rows that link each Q&A turn to its exchange.

In [ ]:
# 1.2 data tests: stop if the release is not what its guide describes
FUSED = re.compile(r"\b(?:do|did|does|would|wo|have|are|should|ca|is|was|has|had|were|could)not\b", re.I)
linked = sent_raw.merge(utt_raw[TURN], on=TURN, how="left", indicator=True)
exch_linked = exch_raw.merge(utt_raw[TURN], on=TURN, how="left", indicator=True)
event_calls = set(sent_raw.loc[sent_raw["call_type"] == "event", ["bank", "quarter"]]
                  .itertuples(index=False, name=None))
earnings = sent_raw[sent_raw["call_type"] == "earnings"].groupby("bank")["quarter"].agg(lambda s: sorted(s.unique()))

checks = {
    "19,092 sentences, 2,570 utterances, 2,473 Q&A exchange rows":
        (len(sent_raw), len(utt_raw), len(exch_raw)) == (19092, 2570, 2473),
    "16 earnings calls per bank, 2022-Q1 to 2025-Q4": all(earnings[b] == QUARTERS for b in BANKS),
    "one event call, JPM 2023-Q2": event_calls == {("JPM", "2023-Q2")},
    "sentence_id unique": sent_raw["sentence_id"].is_unique,
    "cross-release key unique": not sent_raw.duplicated(KEY).any(),
    "no nulls in key columns, text or filler flag":
        sent_raw[KEY + ["section", "speaker_role", "sentence", "is_filler_sentence"]].notna().all().all(),
    "section and role values as documented":
        set(sent_raw["section"]) <= {"prepared", "qa"}
        and set(sent_raw["speaker_role"]) <= {"management", "analyst", "ir", "operator"},
    "text not rewritten (sentence equals sentence_raw)": sent_raw["sentence"].equals(sent_raw["sentence_raw"]),
    "every sentence links to its utterance": linked["_merge"].eq("both").all(),
    "every exchange row links to a Q&A utterance":
        exch_linked["_merge"].eq("both").all() and exch_raw["section"].eq("qa").all(),
    "no fused negations left": not sent_raw["sentence"].str.contains(FUSED).any(),
}
print(pd.Series(checks).map({True: "pass", False: "FAIL"}).to_string())
failed = [name for name, ok in checks.items() if not ok]
if failed:
    raise AssertionError(f"release checks failed: {failed}")

Eleven automatic checks that the data matches its documentation: row counts, 16 earnings calls per bank, one deal call, unique keys, no missing values, valid categories, untouched text and complete links between the tables. All pass; if a later release breaks one, the notebook stops here.

In [ ]:
# 1.3 working columns: year, window, a turn id per utterance, a text key per sentence, and the release's
# Q&A exchange (an analyst's question with the answers to it), numbered within each call
def add_columns(df):
    df = df.copy()
    df["year"] = df["quarter"].str[:4].astype(int)
    df["window"] = df["quarter"].map(WINDOW_OF)
    df["call"] = df["bank"] + " " + df["quarter"] + " " + df["call_type"]
    df["turn_id"] = df[TURN].astype(str).agg("|".join, axis=1)
    return df

sent = add_columns(sent_raw)
utt = add_columns(utt_raw)
sent["key"] = sent[KEY].astype(str).agg("|".join, axis=1)

ex = exch_raw[TURN + ["qa_exchange_id"]].copy()
ex["exchange_id"] = ex[CALL_ID].astype(str).agg("|".join, axis=1) + "|" + ex["qa_exchange_id"].astype(str)
sent = sent.merge(ex[TURN + ["exchange_id"]], on=TURN, how="left", validate="many_to_one")
utt = utt.merge(ex[TURN + ["exchange_id"]], on=TURN, how="left", validate="one_to_one")

per_call = utt.groupby(CALL_ID)["exchange_id"].nunique()
print("Q&A exchanges per call, from the release:")
print(per_call.groupby(["bank", "call_type"]).describe()[["count", "min", "50%", "max"]].to_string())
print("Q&A utterances without an exchange id:", int((utt["section"].eq("qa") & utt["exchange_id"].isna()).sum()))

Adds helper columns to each sentence: year, window, the speaker turn it belongs to, a unique key for joining results with other analyses, and the Q&A exchange from the release. JPMorgan's earnings calls have 27 to 57 exchanges and UBS's 15 to 44; 17 Q&A turns have no exchange number.

In [ ]:
# 1.4 text for the transformers: the raw sentence, minus editorial notes in square brackets ("[indiscernible]",
# "[year-on-year]"). Nothing else is removed: stop words, negations and numbers stay, because the models read them.
# Stop words are used only to name BERTopic's topics (4.1). `text_plain` is the same text with curly quotes and long
# dashes written as plain characters; no model reads it except in the robustness check 4.9.
BRACKET = re.compile(r"\[[^\]]*\]")
TYPOGRAPHY = str.maketrans({"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"', "\u2013": "-", "\u2014": "-",
                            "\u2026": "...", "\u00a0": " "})
sent["had_bracket"] = sent["sentence"].str.contains(BRACKET)
sent["text"] = (sent["sentence"].str.replace(BRACKET, " ", regex=True)
                .str.replace(r"\s+([,.;:?!])", r"\1", regex=True)
                .str.replace(r"\s{2,}", " ", regex=True).str.strip())
sent["text_plain"] = sent["text"].map(lambda s: s.translate(TYPOGRAPHY))
sent["typography_changed"] = sent["text_plain"] != sent["text"]
sent["n_words"] = sent["text"].str.split().str.len().fillna(0).astype(int)
MAX_SHORT = 4          # topic models also drop sentences of 4 words or fewer ("Clear.", "That's helpful.")

# courtesy and procedure sentences the release's filler flag leaves behind ("Thanks for taking my questions.",
# "My second question is on capital.", "Let me take that one."). They are tagged, not removed: every sentence keeps its
# place in the corpus and its join key, and 6.4 shows what the results look like without them.
low, nw = sent["text"].str.lower(), sent["n_words"]
PROCEDURAL = {
    "thanks": low.str.contains(r"\b(?:thanks?|thank you|appreciate[ds]?)\b") & (nw <= 8),
    "question count": (low.str.contains(r"\bquestions?\b|\bfollow[- ]?ups?\b")
                       & low.str.contains(r"\b(?:first|second|third|last|final|one more|two|three|quick|another|couple)\b"
                                          r"|\bfollow[- ]?ups?\b") & (nw <= 9)),
    "routing": low.str.contains(r"\b(?:go ahead|please proceed|line is open|next question|stand ?by|"
                                r"let me (?:ask|take|turn)|move on to|turn it over|pass it (?:on|over)|"
                                r"hand (?:it )?over)\b") & (nw <= 9),
    "praise": ((low.str.contains(r"\b(?:good|fair|great|excellent|interesting|helpful|useful) "
                                 r"(?:question|colou?r|perspective|context|detail)\b") & (nw <= 9))
               | (low.str.contains(r"\b(?:that's|that is|that was|that makes) (?:very |really |super |a |an )*"
                                   r"(?:helpful|clear|great|good|fair|perfect|useful|sense|interesting)\b") & (nw <= 8))),
}
sent["is_procedural"] = np.logical_or.reduce(list(PROCEDURAL.values()))
sent["procedural_rule"] = np.select(list(PROCEDURAL.values()), list(PROCEDURAL), default="")
print("sentences with an editorial bracket:", sent.groupby("bank")["had_bracket"].sum().to_dict())
print("sentences with curly quotes or long dashes (plain-character copy differs):", sent.groupby("bank")["typography_changed"].sum().to_dict())
print("sentences tagged as courtesy or procedure:", sent.groupby("bank")["is_procedural"].sum().to_dict())


Prepares the text for the language models without rewriting it: only editorial notes in square brackets are removed (49 sentences, 39 of them at UBS), and negations, numbers and common words stay, because the models need them to read the meaning. It also tags courtesy and procedure sentences that the release's flag leaves behind (293 in the topic corpus), such as thanks and question counts, without removing them. A copy with plain quotes and dashes is kept only for the robustness check in 4.9.

**One sentence, a different text for each model.** The Sentence-BERT and BERTopic models read the sentence as spoken: only editorial notes in square brackets are taken out. The sentiment models of the other track read a cleaned copy with stop words, negations and numbers removed and the words reduced to their base form. Both start from the same release and the same sentences, so results are always joined on the sentence key (bank, quarter, call type, position in call, sentence number) and never on the text. Cleaning the text differently changes what a model sees, not which sentence a label belongs to; this is why no sentence is deleted here beyond the release's own flag and the four-word rule of 1.5. Section 4.9 shows how sensitive the topic model is even to a change as small as plain quotes.


In [ ]:
# 1.5 filter funnel. The shared base is the release's own filler flag, so every track counts the same
# sentences; the topic track adds one step of its own (short sentences out) and reports it here.
steps = {
    "all sentences": pd.Series(True, index=sent.index),
    "Q&A section only": sent["section"] == "qa",
    "analysts and management (operator, IR out)": sent["speaker_role"].isin(ROLES),
    "release filler flag false (shared base)": ~sent["is_filler_sentence"],
    f"more than {MAX_SHORT} words (topic models)": sent["n_words"] > MAX_SHORT,
}
keep, rows, masks = pd.Series(True, index=sent.index), [], {}
for step, cond in steps.items():
    keep &= cond
    masks[step] = keep.copy()
    n = sent.loc[keep, "bank"].value_counts()
    rows.append({"step": step, **{b: int(n.get(b, 0)) for b in BANKS}, "total": int(keep.sum())})
funnel = pd.DataFrame(rows).set_index("step")
funnel["kept_%"] = (100 * funnel["total"] / funnel["total"].iloc[0]).round(1)

base = sent[masks["release filler flag false (shared base)"]].reset_index(drop=True)
qa = sent[keep].reset_index(drop=True)
display(funnel)
print(f"shared base {len(base):,} sentences · topic corpus {len(qa):,} sentences")
print("from the JPM deal call:", int((qa["call_type"] == "event").sum()))
print(f"courtesy and procedure sentences kept but tagged: {int(qa['is_procedural'].sum()):,} "
      f"({100 * qa['is_procedural'].mean():.1f}% of the topic corpus)")
display(qa.pivot_table(index=["bank", "speaker_role"], columns="window", values="key", aggfunc="count",
                       margins=True).reindex(columns=WINDOW_NAMES + ["All"]))

Builds the corpus step by step: Q&A only, analysts and management only, then the release's own filler flag, which gives a shared base of 10,957 sentences that every model in the team can use. The topic models also drop sentences of four words or fewer, which leaves 10,059: 4,316 from UBS and 5,743 from JPMorgan, 172 of them from JPMorgan's First Republic deal call. Of these, 293 courtesy and procedure sentences are tagged but kept, and the four 2022 quarters (2,670 sentences) give both banks a year before the deals, which the 2023 to 2024 data did not have.

In [ ]:
# 1.6 what the two filters dropped and what the courtesy tag marks. Prints transcript text: clear outputs before committing.
pre = sent[(sent["section"] == "qa") & sent["speaker_role"].isin(ROLES)]
flagged = pre[pre["is_filler_sentence"]]
short = pre[~pre["is_filler_sentence"] & (pre["n_words"] <= MAX_SHORT)]
DIGIT = re.compile(r"\d")
for name, d in (("release filler flag", flagged), (f"{MAX_SHORT} words or fewer, kept by the flag", short)):
    print(f"\n{name}: {len(d):,} sentences, up to {d['n_words'].max()} words, "
          f"{int(d['text'].str.contains(DIGIT).sum())} with a number")
    print(d["text"].value_counts().head(12).to_string())
print("\nshort sentences: 15 random examples")
for s in short["text"].sample(15, random_state=SEED):
    print("   ", s)

tagged = qa[qa["is_procedural"]]
print(f"\ncourtesy and procedure sentences kept but tagged (is_procedural): {len(tagged):,}")
print(tagged["procedural_rule"].value_counts().to_string())
print("share tagged, %, per bank:", (100 * qa.groupby("bank")["is_procedural"].mean()).round(1).to_dict(),
      "| per window:", (100 * qa.groupby("window")["is_procedural"].mean()).round(1).to_dict())
print("15 random tagged examples")
for s in tagged["text"].sample(15, random_state=SEED):
    print("   ", s)
print("\nsentences with curly quotes or long dashes, topic corpus, per bank:",
      qa.groupby("bank")["typography_changed"].agg(["sum", "mean"]).round(3).to_dict("index"))

Shows what the two filters removed. The release's flag drops 2,237 courtesies of one to five words, such as "Thank you." and "Yeah.", none with a number, and the four-word rule drops another 898 short sentences such as "Understood." and "Two questions, please."; 17 of them contain a number. The 293 tagged sentences (3.9% of UBS's corpus and 2.2% of JPMorgan's, spread evenly over the four windows) are mostly thanks and question counts and stay in the corpus.

In [ ]:
# 1.7 save the corpora (parquet, in the outputs folder): the topic corpus and the shared base's keys
cols = ["key", "sentence_id", "utterance_id", "bank", "quarter", "year", "window", "call_type", "call_date",
        "position_in_call", "sentence_number", "speaker_role", "speaker_name", "speaker_institution",
        "turn_id", "exchange_id", "text", "n_words", "is_procedural"]
qa[cols].to_parquet(OUT / "qa_sentences.parquet", index=False)
base[["key", "sentence_id", "bank", "quarter", "call_type", "speaker_role"]].to_csv(OUT / "base_keys.csv", index=False)
funnel.to_csv(OUT / "filter_funnel.csv")
print(f"wrote qa_sentences.parquet ({len(qa):,}), base_keys.csv ({len(base):,}) and filter_funnel.csv")

Saves the topic corpus and the keys of the shared base in the outputs folder, so other analyses can join their results to the same sentences.

### Summary

The 30 September release passes all eleven data checks. Of its 19,092 sentences, 10,957 form the shared base (Q&A, analysts and management, the release's filler flag false), and 10,059 form the topic corpus once sentences of four words or fewer are dropped: UBS 4,316 and JPMorgan 5,743, with JPMorgan's First Republic deal call included (172 sentences). The text reaches the models as spoken, minus 49 editorial notes, and 293 courtesy and procedure sentences are tagged but kept. The four windows hold 2,670 sentences before the deals (2022), 1,379 in the two deal quarters, 2,232 in the year after and 3,778 later. Saved under `notebooks/angelo/outputs/2026-09-30/`: `qa_sentences.parquet`, `base_keys.csv` and `filter_funnel.csv`.

Every later number rests on a checked corpus that other analyses can share and join on. The year before the deals is now in the data, so "before" and "after" can be compared directly.

## Initial data investigation

In [ ]:
# 2.0 chart helpers: 16 quarters on the x axis, the four windows named at the top, the deal quarters shaded,
# one line per bank, breaks as dotted lines
from matplotlib.colors import LinearSegmentedColormap
BLUES = LinearSegmentedColormap.from_list(
    "blues", ["#fcfcfb", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
QLABELS = [q[2:] for q in QUARTERS]                                    # 22-Q1 ...
BREAK_NOTE = "Dotted lines: " + " · ".join(f"{b} {q}, {why}" for b, (q, why) in BREAKS.items())
EVENT_X = QUARTERS.index(EVENT[1]) - 0.3                               # just left of JPM's 2023-Q2 point
X = np.arange(len(QUARTERS))

def quarter_axis(ax):
    ax.set_xticks(X, QLABELS, rotation=45, fontsize=8)
    ax.set_xlim(-0.5, len(QUARTERS) - 0.5)

def mark_windows(ax):
    deal = [QUARTERS.index(q) for q in WINDOWS["deal"]]
    ax.axvspan(deal[0] - 0.5, deal[-1] + 0.5, color="#efeee8", lw=0, zorder=0)
    for w, qs in WINDOWS.items():
        mid = (QUARTERS.index(qs[0]) + QUARTERS.index(qs[-1])) / 2
        ax.text(mid, 1.01, w, transform=ax.get_xaxis_transform(), ha="center", va="bottom",
                fontsize=7, color="#898781")

def mark_breaks(ax, banks=BANKS):
    for b in banks:
        ax.axvline(QUARTERS.index(BREAKS[b][0]), color=BANK_COLOR[b], lw=1, ls=":", zorder=0)

def mark_event(ax, value, label="JPM deal call"):
    # the First Republic deal call alone, as a hollow diamond beside the 2023-Q2 point
    ax.scatter([EVENT_X], [value], marker="D", s=36, facecolors="white", edgecolors=BANK_COLOR["JPM"],
               linewidths=1.5, zorder=4, label=label)

def time_axis(ax, title, banks=BANKS):
    quarter_axis(ax)
    mark_windows(ax)
    mark_breaks(ax, banks)
    ax.set_title(title, loc="left", pad=14)

def bank_lines(ax, series, title):
    # series indexed by (bank, quarter); each line labelled at its last point
    for b in BANKS:
        s = series.xs(b, level="bank").reindex(QUARTERS).astype(float)
        ax.plot(X, s.to_numpy(), color=BANK_COLOR[b], marker="o", ms=3.5, label=b)
    time_axis(ax, title)

No output: defines the chart style. UBS is always blue and JPMorgan orange, the 16 quarters run left to right with the four windows named at the top and the deal quarters shaded, dotted lines mark the two quarters with known breaks in the figures, and a hollow diamond shows JPMorgan's deal call on its own.

In [ ]:
# 2.1 coverage: sentences per call, and the topic corpus per bank and window
coverage = pd.DataFrame({
    "statements": utt.groupby(CALL_ID).size(),
    "sentences": sent.groupby(CALL_ID).size(),
    "qa_sentences": sent[sent["section"] == "qa"].groupby(CALL_ID).size(),
    "shared_base": base.groupby(CALL_ID).size(),
    "topic_corpus": qa.groupby(CALL_ID).size(),
}).fillna(0).astype(int)
print("topic corpus per call")
display(coverage["topic_corpus"].unstack("quarter").reindex(columns=QUARTERS))

by_window = qa.groupby(["bank", "window"]).agg(calls=("call", "nunique"), sentences=("key", "size"),
                                               analyst_pct=("speaker_role", lambda r: 100 * (r == "analyst").mean()))
by_window["sentences_per_call"] = by_window["sentences"] / by_window["calls"]
display(by_window.reindex(pd.MultiIndex.from_product([BANKS, WINDOW_NAMES])).round(1))

Sentences per call and per window. UBS's Q&A shrinks over time, from about 341 clean sentences per call in 2022 to 222 in the later window, while JPMorgan's grows from 327 to 407. Analysts give about a third of UBS's sentences and a quarter to under a third of JPMorgan's.

In [ ]:
# 2.2 filter funnel with reasons
fig, ax = plt.subplots(figsize=(8.5, 3.6))
y, h = np.arange(len(funnel)), 0.38
for i, b in enumerate(BANKS):
    ys = y + (i - 0.5) * h
    ax.barh(ys, funnel[b], height=h - 0.05, color=BANK_COLOR[b], label=b)
    for yy, v in zip(ys, funnel[b]):
        ax.text(v, yy, f" {v:,}", va="center", fontsize=8, color="#52514e")
ax.set_yticks(y, funnel.index)
ax.invert_yaxis()
ax.grid(axis="y", visible=False)
ax.set_xlabel("sentences")
ax.legend(loc="lower right")
ax.set_title(f"From {funnel['total'].iloc[0]:,} sentences to a "
             f"{funnel['total'].iloc[-1]:,}-sentence topic corpus, 2022 to 2025", loc="left")
plt.tight_layout()
fig.savefig(FIG / "fig_2.2_funnel.png", dpi=150, bbox_inches="tight")
plt.show()

Charts the filtering steps for each bank, from 19,092 sentences to the 10,059-sentence topic corpus.

In [ ]:
# 2.3 Q&A volume per call: analysts, exchanges with a clean analyst sentence, and how much management says
# per exchange. Lines: the earnings calls; diamond: the JPM deal call on its own.
turns = utt[utt["section"] == "qa"]
asked = qa[qa["speaker_role"] == "analyst"]
volume = pd.DataFrame({
    "analysts": turns[turns["speaker_role"] == "analyst"].groupby(CALL_ID)["speaker_name"].nunique(),
    "exchanges": asked.groupby(CALL_ID)["exchange_id"].nunique(),
    "clean Q&A sentences": qa.groupby(CALL_ID).size(),
    "management sentences": qa[qa["speaker_role"] == "management"].groupby(CALL_ID).size(),
})
volume["management sentences per exchange"] = volume["management sentences"] / volume["exchanges"]
panels = ["analysts", "exchanges", "clean Q&A sentences", "management sentences per exchange"]
fig, axes = plt.subplots(2, 2, figsize=(11, 6.2))
for ax, col in zip(axes.flat, panels):
    bank_lines(ax, volume[col].xs("earnings", level="call_type"), col)
    mark_event(ax, volume.loc[EVENT, col])
    ax.set_ylim(bottom=0)
axes[0, 0].legend(loc="lower left", fontsize=8)
fig.text(0.01, 0.005, BREAK_NOTE + "\n" + EVENT_NOTE + " · shaded: the two deal quarters",
         fontsize=8, color="#52514e")
plt.tight_layout(rect=(0, 0.05, 1, 1))
fig.savefig(FIG / "fig_2.3_qa_volume.png", dpi=150, bbox_inches="tight")
plt.show()
display(volume.groupby(["bank", "call_type"]).mean().round(1))
print("clean Q&A sentences per year:", qa.groupby(["bank", "year"]).size().unstack().to_dict("index"))

Q&A volume per call: both banks take questions from 7 to 16 analysts. UBS has fewer exchanges than JPMorgan (15 against 26 per call on average) and, from 2024, fewer clean sentences, down to 132 in 2025-Q2, while JPMorgan's volume grows in 2025.

In [ ]:
# 2.4 keyword baseline for merger talk, per quarter. Before 2023 a hit can be another deal (an acquisition
# that was planned or done elsewhere), so this is a rough count, not a measure of the merger.
MERGER_KW = re.compile(r"(?i:credit suisse|first republic|\bintegrat\w*|\bacquisi\w*|\bmerger\w*)|\bCS\b|\bFRC\b")
qa["merger_keyword"] = qa["text"].str.contains(MERGER_KW)
kw_share = 100 * qa[qa["call_type"] == "earnings"].groupby(["bank", "quarter"])["merger_keyword"].mean()
fig, ax = plt.subplots(figsize=(9, 3.4))
bank_lines(ax, kw_share, "Share of clean Q&A sentences with a merger keyword, %")
mark_event(ax, 100 * qa.loc[qa["call_type"] == "event", "merger_keyword"].mean())
ax.set_ylim(bottom=0)
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
fig.savefig(FIG / "fig_2.4_merger_keywords.png", dpi=150, bbox_inches="tight")
plt.show()
kw = qa.pivot_table(index="bank", columns="year", values="merger_keyword", aggfunc="sum")
kw["share_% all years"] = 100 * qa.groupby("bank")["merger_keyword"].mean()
display(kw.round(1))
print("2022 hits, UBS:")
for s in qa.loc[(qa["bank"] == "UBS") & (qa["year"] == 2022) & qa["merger_keyword"], "text"]:
    print("   ", s[:170])

A simple keyword search for merger talk ("Credit Suisse", "First Republic", "integration", "acquisition", "merger"). At UBS it jumps from 4 sentences in 2022 to 101 in 2023 and stays at 44 to 45 a year after; at JPMorgan it peaks at 48 in 2023. The 2022 hits at UBS are about a different deal, the planned purchase of Wealthfront, and this rough count is the baseline that the topic model and the theme tagger are checked against.

In [ ]:
# 2.5 answer length: words per management answer and clean sentences per answer, by bank and year
answers = utt[(utt["section"] == "qa") & (utt["speaker_role"] == "management")].copy()
answers["words"] = answers["text"].str.split().str.len()
length = answers.groupby(["bank", "year"]).agg(answers=("words", "size"), median_words=("words", "median"))
length["clean_sentences_per_answer"] = (qa[qa["speaker_role"] == "management"].groupby(["bank", "year", "turn_id"])
                                        .size().groupby(["bank", "year"]).mean())
display(length.unstack("bank").round(1))

Answer length: UBS management answers are long and get longer (median 117 words in 2022, 245 in 2025), while JPMorgan's are shorter (38 to 104 words). Each management answer gives 6 to 10 clean sentences, so working at sentence level keeps long answers from being cut off at the models' input limit.

### Summary

All 33 calls are in the corpus. The two banks run their Q&A differently. JPMorgan has more exchanges per call (26 against 15) and its Q&A grows to 2025, while UBS's shrinks from 1,364 clean sentences in 2022 to 867 in 2025 and its management answers get longer (median 117 to 245 words). Analysts give about a third of UBS's sentences and about a quarter of JPMorgan's. Keyword search finds merger talk mostly at UBS: 4 sentences in 2022, 101 in 2023, then 44 or 45 a year; JPMorgan peaks at 48 in 2023.

From here on the banks are compared on shares, not counts, because their Q&A volumes move in opposite directions.

## Themes with a fine-tuned Sentence-BERT tagger

The first model reads every sentence against seven fixed themes. Each sentence takes the theme whose example sentences it is closest to in meaning, so the same question is asked of every quarter and the shares can be tracked over time.

| Label | Theme | A sentence belongs here when it is mainly about |
|---|---|---|
| `integration` | Integration and M&A | the 2023 acquisition, integration progress and milestones, client or system migration, legal-entity mergers, deal accounting |
| `restructuring` | Restructuring | winding down legacy or non-core books, exiting businesses, reorganisation, restructuring charges |
| `capital` | Capital and liquidity | capital ratios and requirements, buybacks and dividends, RWA and leverage, liquidity, funding, deposit outflows |
| `risk` | Risk and credit quality | credit losses and reserves, loan quality, real-estate exposure, market and rate risk, litigation |
| `costs` | Costs and efficiency | expenses, cost savings, cost/income, headcount, investment spend |
| `revenue` | Revenue and outlook | net interest income and deposit pricing, fees, trading, wealth flows, guidance, client activity |
| `other` | Other | none of the above, or procedure |

Step 3.2 turns the codebook into eight example sentences per theme, none of which names either bank. These 56 examples are the only labelled data in the notebook: they define the themes, the two Sentence-BERT models are fine-tuned on them, and the fine-tuning is checked on them. No transcript sentence is labelled by hand or by a language model, and the fine-tuning never sees a transcript sentence, so it does not depend on the data release.

**Fine-tuning** follows SetFit, a standard recipe for fine-tuning a Sentence-BERT model on a handful of labelled examples. Every example is paired 20 times with an example of the same theme (target similarity 1) and 20 times with an example of a different theme (target 0), 2,240 pairs. The whole network is trained for one pass over the pairs with the cosine-similarity loss (learning rate 2e-5, batches of 16, 10% warm-up), so that sentences on the same theme move closer together. These are SetFit's default settings, fixed before any result was seen. The checks need no labelled transcript sentence:

- **Held-out examples (3.3).** Four-fold cross-validation: the model is fine-tuned on six examples per theme and has to place the two it never saw. The pre-trained model is scored the same way, so the difference is the effect of fine-tuning.
- **Stability (3.3).** Each of the four fold models tags the whole corpus. If the tags barely change when a different quarter of the examples is left out, the result does not hang on the wording of a few examples.
- **Agreement between the models (3.4).** MiniLM and mpnet are different networks; if fine-tuning makes them agree more often on real sentences, the themes are better defined.
- **Merger sentences (3.4).** Sentences with a merger keyword (2.4) should mostly land on "integration".

**Rule for the final model, fixed in advance:** the model that places more held-out examples correctly after fine-tuning; if the two are within two examples of each other, the more stable one.

In [ ]:
# 3.1 sentence embeddings with two pre-trained Sentence-BERT models, cached next to the row keys and the text
# fingerprints they belong to (a cache_key is the sentence key plus a hash of its text, so changed text is never
# served from an old cache).
# These are the vectors before fine-tuning (3.4 adds the fine-tuned ones). MiniLM also feeds BERTopic (Section 4).
import hashlib
from sentence_transformers import SentenceTransformer

qa["cache_key"] = [f"{k}#{hashlib.md5(t.encode()).hexdigest()[:8]}" for k, t in zip(qa["key"], qa["text"])]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODELS = {"MiniLM": "sentence-transformers/all-MiniLM-L6-v2",
          "mpnet": "sentence-transformers/all-mpnet-base-v2"}

def free_gpu():
    # a SentenceTransformer holds a reference to itself, so `del` alone leaves its weights on the GPU
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

def encode(model, texts, batch_size=64):
    return model.encode([str(t) for t in texts], batch_size=batch_size, normalize_embeddings=True,
                        convert_to_numpy=True, show_progress_bar=False)

def cached(name, keys):
    # vectors saved by an earlier run, used only if they belong to exactly these sentences in this order
    path, key_path = OUT / f"emb_qa_{name}.npy", OUT / f"emb_qa_{name}_keys.csv"
    if path.exists() and key_path.exists() and pd.read_csv(key_path)["key"].tolist() == list(keys):
        return np.load(path)
    return None

def save_vectors(name, E, keys):
    np.save(OUT / f"emb_qa_{name}.npy", E)
    pd.Series(list(keys), name="key").to_csv(OUT / f"emb_qa_{name}_keys.csv", index=False)   # keys here are cache keys

def embed(tag, texts, keys, batch_size=64):
    E = cached(tag, keys)
    if E is not None:
        print(f"{tag}: loaded from emb_qa_{tag}.npy")
        return E
    model = SentenceTransformer(MODELS[tag], device=DEVICE)
    t0 = time.time()
    E = encode(model, texts, batch_size)
    print(f"{tag}: {E.shape[0]:,} x {E.shape[1]} in {time.time() - t0:.0f}s on {DEVICE}, "
          f"max_seq_length {model.max_seq_length}")
    save_vectors(tag, E, keys)
    del model
    free_gpu()
    return E

emb = {tag: embed(tag, qa["text"], qa["cache_key"], batch_size=64 if tag == "MiniLM" else 32) for tag in MODELS}
for tag, E in emb.items():
    norms = np.linalg.norm(E, axis=1)
    print(f"{tag:7s} {E.shape[0]:,} x {E.shape[1]}   norms {norms.min():.3f} to {norms.max():.3f}")

Turns each of the 10,059 sentences into a vector with two pre-trained Sentence-BERT models, MiniLM (384 numbers per sentence) and mpnet (768), and saves them together with a fingerprint of each sentence's text, so a later run reuses vectors only when the text is exactly the same. Encoding takes 8 seconds with MiniLM and about a minute with mpnet on the laptop GPU.

In [ ]:
# 3.2 theme examples: eight bank-neutral sentences per theme. "other" has its own examples, so a sentence with no
# business topic lands there by similarity; there is no cut-off. A sentence takes the theme whose centroid (the
# average of its eight examples) is closest. This cell scores the pre-trained models, before fine-tuning.
from sklearn.metrics import cohen_kappa_score

EXAMPLES = {
    "integration": [
        "The integration of the acquired bank is progressing well and we remain on track with our milestones.",
        "We are migrating the acquired clients and accounts onto our own platforms and systems.",
        "The merger of the two legal entities was completed after the transaction closed.",
        "Purchase accounting, negative goodwill and other acquisition-related effects from the deal.",
        "How is the combination of the two businesses going, and what are the remaining integration risks?",
        "When will the legal entities be merged, and when will the acquired clients be on one platform?",
        "How much of the integration work is still ahead of you, and what are the next milestones?",
        "Most clients of the acquired bank have stayed with us since the takeover.",
    ],
    "restructuring": [
        "We continue to wind down the legacy portfolio and exit businesses that do not fit our strategy.",
        "Running off non-core positions released risk-weighted assets and reduced the balance sheet.",
        "The reorganisation involves closing overlapping units and simplifying the group structure.",
        "Restructuring charges and exit costs were higher this quarter.",
        "How much longer will it take to wind down the remaining legacy book?",
        "We agreed to sell a business that is no longer core to the group.",
        "The run-down of the non-core unit is ahead of plan, with fewer positions left to exit.",
        "We are reorganising our divisions into a simpler structure with fewer layers of management.",
    ],
    "capital": [
        "Our CET1 capital ratio remains well above the regulatory requirement.",
        "We intend to return excess capital to shareholders through share buybacks and dividends.",
        "The proposed capital rules and Basel III requirements would increase the capital we need to hold.",
        "Liquidity and funding remain strong, with a comfortable liquidity coverage ratio.",
        "Risk-weighted assets and the leverage ratio determine how much balance sheet we can deploy.",
        "When do you expect to restart share buybacks, and at what size?",
        "How do you think about the payout split between dividends and buybacks over the next few years?",
        "The capital surcharge and the stress capital buffer set how much capital we must hold.",
    ],
    "risk": [
        "Credit quality remains solid, but we expect net charge-offs to normalise from low levels.",
        "We added to our loan loss reserves because the macroeconomic outlook weakened.",
        "Exposure to commercial real estate and office loans is an area we are watching closely.",
        "Litigation, legal provisions and regulatory investigations remain a risk to earnings.",
        "Market volatility and geopolitical uncertainty could affect the portfolio.",
        "Rates staying higher for longer could strain borrowers and slow the economy.",
        "Card net charge-offs rose as consumer credit continued to normalise.",
        "Higher rates have caused unrealised losses in the securities portfolio.",
    ],
    "costs": [
        "We are on track to deliver the planned gross cost savings.",
        "Expenses rose this quarter, driven by investment in technology and higher compensation.",
        "Our cost-to-income ratio should improve as the efficiency measures come through.",
        "We reduced headcount and contractor spend to bring the cost base down.",
        "What is your expense guidance for next year?",
        "We expect adjusted expenses for next year to be roughly in line with this year.",
        "Compensation costs went up because of higher performance-related pay.",
        "Technology, marketing and real estate were the main drivers of the increase in non-compensation expenses.",
    ],
    "revenue": [
        "Net interest income will come down as deposit costs rise and balances move to higher-yielding products.",
        "Investment banking fees and markets revenue were strong compared with last year.",
        "Wealth management attracted net new assets and fee income grew.",
        "Our outlook assumes modest loan growth and a few rate cuts next year.",
        "Client activity and consumer spending remain healthy.",
        "Net interest income guidance for next year depends on the path of rate cuts and the forward curve.",
        "We are winning back market share and client flows are returning.",
        "What is your outlook for revenue growth and returns over the next few years?",
    ],
    "other": [
        "Thank you, that is very helpful.",
        "Let me take your second question.",
        "I would just add one point to that.",
        "What exactly do you mean by that?",
        "We are not going to comment on speculation.",
        "That is a fair question, and we will come back to it.",
        "We will give you more detail on that at a later date.",
        "I think that is broadly right, and we will see how it plays out.",
    ],
}
THEMES = list(EXAMPLES)
BANK_NAMES = re.compile(r"\b(?:ubs|jp ?morgan|chase|credit suisse|first republic|swiss|switzerland)\b", re.I)
assert not any(BANK_NAMES.search(s) for v in EXAMPLES.values() for s in v), "an example names a bank"
assert all(len(v) == 8 for v in EXAMPLES.values()), "eight examples per theme"
ex_text = np.array([s for v in EXAMPLES.values() for s in v])
ex_label = np.array([t for t, v in EXAMPLES.items() for _ in v])

def centroids(E_ex, labels):
    C = np.vstack([E_ex[labels == t].mean(axis=0) for t in THEMES])
    return C / np.linalg.norm(C, axis=1, keepdims=True)

def tag_themes(E, C):
    # nearest theme centroid, and the margin over the runner-up (how clear-cut the choice is)
    S = E @ C.T
    top2 = np.sort(S, axis=1)[:, -2:]
    return np.array(THEMES)[S.argmax(axis=1)], top2[:, 1] - top2[:, 0]

E_ex_base = {}
for tag in MODELS:
    model = SentenceTransformer(MODELS[tag], device=DEVICE)
    E_ex_base[tag] = encode(model, ex_text)
    del model
    free_gpu()
    qa[f"theme_base_{tag}"], qa[f"margin_base_{tag}"] = tag_themes(emb[tag], centroids(E_ex_base[tag], ex_label))

print("theme shares by bank before fine-tuning (% of sentences)")
display(pd.concat({tag: pd.crosstab(qa[f"theme_base_{tag}"], qa["bank"], normalize="columns").mul(100)
                   for tag in MODELS}, axis=1).reindex(THEMES).fillna(0).round(1))
same = qa["theme_base_MiniLM"] == qa["theme_base_mpnet"]
print(f"MiniLM and mpnet give the same theme for {100 * same.mean():.0f}% of sentences "
      f"(kappa {cohen_kappa_score(qa['theme_base_MiniLM'], qa['theme_base_mpnet']):.2f})")

Defines the seven themes with eight example sentences each and tags every sentence with the nearest theme before any fine-tuning. The pre-trained models already show bank differences (with MiniLM, risk is 21% of JPMorgan's sentences against 10% of UBS's, and integration and restructuring are more frequent at UBS). The two models give the same theme for only 65% of sentences (kappa 0.56), so the untuned themes are not yet well defined.

In [ ]:
# 3.3 fine-tuning, and four-fold cross-validation on the theme examples: each fold fine-tunes on 6 examples per
# theme and places the 2 held out; the pre-trained model is scored the same way. Each fold's tagger also tags the
# whole corpus, to see how much the tags depend on which examples were used. 8 fine-tuning runs; results are
# saved, and a later run reloads them when the corpus is unchanged.
import itertools
from sentence_transformers.sentence_transformer import losses
from sentence_transformers.util import batch_to_device
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

FT = {"pairs_per_example": 20, "epochs": 1, "batch_size": 16, "lr": 2e-5, "warmup": 0.1}   # SetFit defaults

def example_pairs(texts, labels, rng):
    # every example gets pairs_per_example same-theme partners (target 1) and as many other-theme ones (target 0)
    pairs, idx = [], np.arange(len(labels))
    for _ in range(FT["pairs_per_example"]):
        for i, t in enumerate(labels):
            same, other = idx[(labels == t) & (idx != i)], idx[labels != t]
            pairs += [(texts[i], texts[rng.choice(same)], 1.0), (texts[i], texts[rng.choice(other)], 0.0)]
    return pairs

def fine_tune(tag, texts, labels, seed=SEED):
    # SetFit's first stage: contrastive training of the whole Sentence-BERT model on example pairs.
    # Returns the model and the loss of every training step.
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = SentenceTransformer(MODELS[tag], device=DEVICE)
    loss_fn = losses.CosineSimilarityLoss(model)
    pairs = example_pairs([str(t) for t in texts], np.asarray(labels), rng)
    steps = FT["epochs"] * int(np.ceil(len(pairs) / FT["batch_size"]))
    opt = torch.optim.AdamW(model.parameters(), lr=FT["lr"])
    sched = get_linear_schedule_with_warmup(opt, int(FT["warmup"] * steps), steps)
    model.train()
    log = []
    for _ in range(FT["epochs"]):
        order = rng.permutation(len(pairs))
        for start in range(0, len(pairs), FT["batch_size"]):
            batch = [pairs[i] for i in order[start:start + FT["batch_size"]]]
            a, b = (batch_to_device(model.preprocess([p[k] for p in batch]), model.device) for k in (0, 1))
            y = torch.tensor([p[2] for p in batch], dtype=torch.float32, device=model.device)
            loss = loss_fn([a, b], y)
            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()
            log.append(loss.item())
    opt.zero_grad(set_to_none=True)                        # drop the last gradients, no longer needed
    model.eval()
    return model, log

folds = list(StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED).split(ex_text, ex_label))
CV_PATH, FOLD_PATH = OUT / "finetune_cv_examples.csv", OUT / "finetune_fold_tags.csv"
fold_file = pd.read_csv(FOLD_PATH) if FOLD_PATH.exists() else None
if CV_PATH.exists() and fold_file is not None and fold_file["key"].tolist() == qa["cache_key"].tolist():
    cv = pd.read_csv(CV_PATH)
    fold_tags = {(c.split("|")[0], c.split("|")[1], int(c.split("|")[2])): fold_file[c].to_numpy()
                 for c in fold_file.columns if c != "key"}
    print(f"cross-validation loaded from {CV_PATH.name} and {FOLD_PATH.name}")
else:
    cv_rows, fold_tags = [], {}
    t0 = time.time()
    for tag in MODELS:
        for k, (tr, te) in enumerate(folds):
            C = centroids(E_ex_base[tag][tr], ex_label[tr])                     # before: pre-trained model
            cv_rows += [{"model": tag, "stage": "before", "fold": k, "example": i, "theme": ex_label[i],
                         "predicted": p} for i, p in zip(te, tag_themes(E_ex_base[tag][te], C)[0])]
            fold_tags[tag, "before", k] = tag_themes(emb[tag], C)[0]
            model, _ = fine_tune(tag, ex_text[tr], ex_label[tr], seed=SEED + k)  # after: fine-tuned on this fold
            C = centroids(encode(model, ex_text[tr]), ex_label[tr])
            cv_rows += [{"model": tag, "stage": "after", "fold": k, "example": i, "theme": ex_label[i],
                         "predicted": p} for i, p in zip(te, tag_themes(encode(model, ex_text[te]), C)[0])]
            fold_tags[tag, "after", k] = tag_themes(encode(model, qa["text"], batch_size=128), C)[0]
            del model
            free_gpu()
    print(f"{len(MODELS) * len(folds)} fine-tuning runs in {(time.time() - t0) / 60:.1f} min on {DEVICE}")
    cv = pd.DataFrame(cv_rows)
    cv["correct"] = cv["theme"] == cv["predicted"]
    cv.to_csv(CV_PATH, index=False)
    pd.DataFrame({"key": qa["cache_key"], **{f"{t}|{s}|{k}": v for (t, s, k), v in fold_tags.items()}}).to_csv(
        FOLD_PATH, index=False)

boot = np.random.default_rng(SEED).integers(0, len(ex_label), size=(1000, len(ex_label)))
cv_summary = []
for (tag, stage), g in cv.groupby(["model", "stage"], sort=False):
    ok = g.sort_values("example")["correct"].to_numpy()
    lo, hi = np.percentile(100 * ok[boot].mean(axis=1), [2.5, 97.5])
    runs = [fold_tags[tag, stage, k] for k in range(len(folds))]
    cv_summary.append({"model": tag, "stage": stage, "held-out right": f"{ok.sum()} of {len(ok)}",
                       "accuracy %": 100 * ok.mean(), "95% low": lo, "95% high": hi,
                       "macro-F1": f1_score(g["theme"], g["predicted"], average="macro"),
                       "tags unchanged across folds %":
                           100 * np.mean([(a == b).mean() for a, b in itertools.combinations(runs, 2)])})
cv_summary = pd.DataFrame(cv_summary).set_index(["model", "stage"])
display(cv_summary.round(2))
print("held-out examples placed correctly, per theme (of 8)")
display(cv.pivot_table(index="theme", columns=["model", "stage"], values="correct", aggfunc="sum")
        .reindex(index=THEMES, columns=pd.MultiIndex.from_product([list(MODELS), ["before", "after"]])))

Fine-tunes each model on the examples while holding out a quarter of them each time (8 runs, 11 minutes), then checks how many held-out examples it places correctly. Fine-tuning clearly helps mpnet (43 to 50 of 56 right, 89%) and barely helps MiniLM (44 to 45). The tags of the real sentences stay about as stable when other examples are used (72% to 77% unchanged), so the themes do not hang on a few examples.

In [ ]:
# 3.4 the final fine-tuning: each model trained on all 56 examples, then the whole corpus tagged again. Before and
# after on real transcript sentences, the chart for the fine-tuning slide, and the final model by the rule above.
# Each fine-tuned model, its vectors and its theme centroids are saved; a later run reloads them.
loss_log, C_ft = {}, {}
for tag in MODELS:
    name = f"{tag}_finetuned"
    E = cached(name, qa["cache_key"])
    c_path, l_path = OUT / f"centroids_{name}.npy", OUT / f"finetune_loss_{tag}.csv"
    if E is not None and c_path.exists() and l_path.exists():
        C_ft[tag], loss_log[tag] = np.load(c_path), pd.read_csv(l_path)["loss"].tolist()
        print(f"{tag}: fine-tuned vectors loaded from emb_qa_{name}.npy")
    else:
        t0 = time.time()
        model, loss_log[tag] = fine_tune(tag, ex_text, ex_label)
        E = encode(model, qa["text"], batch_size=128)
        C_ft[tag] = centroids(encode(model, ex_text), ex_label)
        model.save(str(OUT / f"sbert_themes_{tag}"))
        save_vectors(name, E, qa["cache_key"])
        np.save(c_path, C_ft[tag])
        pd.Series(loss_log[tag], name="loss").to_csv(l_path, index=False)
        del model
        free_gpu()
        print(f"{tag}: {len(loss_log[tag])} steps in {time.time() - t0:.0f}s")
    print(f"{tag}: training loss {np.mean(loss_log[tag][:10]):.3f} (first 10 steps) to "
          f"{np.mean(loss_log[tag][-10:]):.3f} (last 10)")
    emb[name] = E
    qa[f"theme_ft_{tag}"], qa[f"margin_ft_{tag}"] = tag_themes(E, C_ft[tag])

rows = []
for stage, p in (("before", "base"), ("after", "ft")):
    both = qa[f"theme_{p}_MiniLM"] == qa[f"theme_{p}_mpnet"]
    for tag in MODELS:
        t = qa[f"theme_{p}_{tag}"]
        rows.append({"model": tag, "stage": stage,
                     "held-out examples right %": cv_summary.loc[(tag, stage), "accuracy %"],
                     "tags unchanged across folds %": cv_summary.loc[(tag, stage), "tags unchanged across folds %"],
                     "same theme as the other model %": 100 * both.mean(),
                     "merger sentences tagged integration %": 100 * (t[qa["merger_keyword"]] == "integration").mean(),
                     "other %": 100 * (t == "other").mean(),
                     "median margin": qa[f"margin_{p}_{tag}"].median()})
diag = pd.DataFrame(rows).set_index(["model", "stage"])
display(diag.round(1))
for stage, p in (("before", "base"), ("after", "ft")):
    print(f"MiniLM against mpnet, {stage} fine-tuning: kappa "
          f"{cohen_kappa_score(qa[f'theme_{p}_MiniLM'], qa[f'theme_{p}_mpnet']):.2f}")

acc = cv[cv["stage"] == "after"].groupby("model")["correct"].sum()
best, runner = acc.idxmax(), acc.idxmin()
CHOSEN = best if acc[best] - acc[runner] > 2 else diag.xs("after", level="stage")["tags unchanged across folds %"].idxmax()
print(f"held-out examples right after fine-tuning: {acc.to_dict()} of {len(ex_label)}  ->  chosen model: {CHOSEN}")

# chart: before (hollow, grey) and after (filled) for each check; lines are 95% intervals
STAGE_STYLE = {"before": {"facecolors": "white", "edgecolors": "#898781"},
               "after": {"facecolors": "#4a3aa7", "edgecolors": "#4a3aa7"}}
OFFSET = {"before": 0.14, "after": -0.14}

def before_after_dots(ax, values, title, intervals=None):
    # values: {row label: {stage: value}}; intervals: {row label: {stage: (low, high)}}
    for y, (label, v) in enumerate(values.items()):
        for stage in ("before", "after"):
            yy = y + OFFSET[stage]
            if intervals:
                lo, hi = intervals[label][stage]
                ax.plot([lo, hi], [yy, yy], color=STAGE_STYLE[stage]["edgecolors"], lw=1.2, zorder=2)
            ax.scatter([v[stage]], [yy], s=64, linewidths=1.6, zorder=3, label=stage if y == 0 else None,
                       **STAGE_STYLE[stage])
            up = stage == "after"                          # after-labels above, before-labels below
            ax.annotate(f"{v[stage]:.0f}", (v[stage], yy), xytext=(0, 7 if up else -8), textcoords="offset points",
                        ha="center", va="bottom" if up else "top", fontsize=8, color="#52514e")
    ax.set_yticks(range(len(values)), list(values))
    ax.set_ylim(len(values) - 0.5, -0.5)
    ax.grid(axis="y", visible=False)
    ax.set_xlim(0, 105)
    ax.set_xlabel("%")
    ax.set_title(title, loc="left")

def per_model(col):
    return {t: {s: diag.loc[(t, s), col] for s in ("before", "after")} for t in MODELS}

ci = {t: {s: tuple(cv_summary.loc[(t, s), ["95% low", "95% high"]]) for s in ("before", "after")} for t in MODELS}
fig, axes = plt.subplots(1, 4, figsize=(13, 2.9))
before_after_dots(axes[0], per_model("held-out examples right %"), "Held-out examples\nplaced correctly", ci)
before_after_dots(axes[1], per_model("tags unchanged across folds %"), "Corpus tags unchanged when\nother examples are used")
before_after_dots(axes[2], per_model("merger sentences tagged integration %"), "Merger sentences\ntagged integration")
before_after_dots(axes[3], {"MiniLM\nand mpnet": per_model("same theme as the other model %")["MiniLM"]},
                  "Both models give\nthe same theme")
axes[0].legend(loc="lower left", fontsize=8)
fig.suptitle("Sentence-BERT fine-tuning: before (hollow) and after (filled)", x=0.01, ha="left", fontweight="bold")
fig.text(0.01, 0.005, "Held-out examples: four-fold cross-validation on the 56 theme examples; lines are 95% "
         "intervals. The other panels use all clean Q&A sentences, 2022 to 2025.", fontsize=8, color="#52514e")
plt.tight_layout(rect=(0, 0.05, 1, 0.95))
fig.savefig(FIG / "fig_3.4_finetuning.png", dpi=150, bbox_inches="tight")
plt.show()

qa["theme"] = qa[f"theme_ft_{CHOSEN}"]
qa[["key", "sentence_id", "call_type", "theme"] + [f"theme_{p}_{t}" for p in ("base", "ft") for t in MODELS]] \
    .to_csv(OUT / "themes_sentence.csv", index=False)
print(f"wrote themes_sentence.csv ({len(qa):,} sentences); fine-tuned models in sbert_themes_<model>/")

Fine-tunes both models on all 56 examples and picks mpnet by the rule fixed in advance, because it places 50 of 56 held-out examples against 45 for MiniLM. On real sentences the effect is modest: merger-keyword sentences tagged integration rise from 37% to 48% with mpnet, but the two models agree on slightly fewer sentences (63%, from 65%). The themes of every sentence are saved, and the chart shows each check before and after.

In [ ]:
# 3.5 theme shares per bank and quarter (fine-tuned tagger), with 95% intervals from resampling whole utterances
# (sentences of one answer are not independent, so the answer is the unit that is resampled). JPM 2023-Q2 pools the
# earnings call and the deal call; the diamond shows the deal call alone.
B = 1000

def resample_weights(rng, n_units, n=B):
    # n bootstrap samples as counts: row i says how often each unit was drawn in sample i
    W = np.zeros((n, n_units))
    np.add.at(W, (np.arange(n)[:, None], rng.integers(0, n_units, size=(n, n_units))), 1)
    return W

def unit_counts(df, col, cats):
    return pd.crosstab(df["turn_id"], df[col]).reindex(columns=cats, fill_value=0).to_numpy()

def shares(df, col, cats, rng):
    # point shares per category, and B bootstrap replicates from resampled utterances
    counts = unit_counts(df, col, cats)
    boot = resample_weights(rng, len(counts)) @ counts
    return counts.sum(axis=0) / counts.sum(), boot / boot.sum(axis=1, keepdims=True)

rng = np.random.default_rng(SEED)
records = []
for (b, q), g in qa.groupby(["bank", "quarter"]):
    point, boot = shares(g, "theme", THEMES, rng)
    lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
    records += [{"bank": b, "quarter": q, "theme": t, "share": 100 * point[k], "lo": 100 * lo[k],
                 "hi": 100 * hi[k]} for k, t in enumerate(THEMES)]
prevalence = pd.DataFrame(records)
prevalence.round(2).to_csv(OUT / "theme_shares_bank_quarter.csv", index=False)
deal_themes = pd.Series(100 * shares(qa[qa["call_type"] == "event"], "theme", THEMES, rng)[0], index=THEMES)

fig, axes = plt.subplots(2, 3, figsize=(13, 6.6), sharex=True)
for ax, t in zip(axes.flat, [t for t in THEMES if t != "other"]):
    p = prevalence[prevalence["theme"] == t].set_index(["bank", "quarter"])
    for b in BANKS:
        s = p.xs(b, level="bank").reindex(QUARTERS)
        ax.fill_between(X, s["lo"], s["hi"], color=BANK_COLOR[b], alpha=0.12, lw=0)
        ax.plot(X, s["share"], color=BANK_COLOR[b], marker="o", ms=3, label=b)
    mark_event(ax, deal_themes[t])
    time_axis(ax, t)
    ax.set_ylim(bottom=0)
for ax in axes[:, 0]:
    ax.set_ylabel("% of the bank's Q&A sentences")
fig.legend(*axes[0, 0].get_legend_handles_labels(), loc="upper right", ncol=3)
fig.suptitle("Theme shares per quarter, UBS against JPMorgan, 2022 to 2025", x=0.01, ha="left", fontweight="bold")
fig.text(0.01, 0.005, BREAK_NOTE + " · Shaded: 95% interval from resampling whole answers · grey: deal quarters\n"
         + EVENT_NOTE + "; JPM's 2023-Q2 line pools it with the earnings call", fontsize=8, color="#52514e")
plt.tight_layout(rect=(0, 0.05, 1, 0.96))
fig.savefig(FIG / "fig_3.5_theme_shares.png", dpi=150, bbox_inches="tight")
plt.show()

print("theme share by bank and window (% of sentences)")
display(qa.pivot_table(index="theme", columns=["bank", "window"], values="key", aggfunc="count")
        .div(qa.groupby(["bank", "window"]).size(), axis=1).mul(100).round(1)
        .reindex(index=THEMES, columns=pd.MultiIndex.from_product([BANKS, WINDOW_NAMES])).fillna(0))

Tracks the theme shares per quarter with 95% ranges. At UBS integration rises from 2.8% of the Q&A in 2022 to 12.1% in the deal quarters and stays at 7% to 8% afterwards, and restructuring goes from 1.6% to 10.9% and then about 5%, while JPMorgan's integration share stays at 2% to 4%. JPMorgan's risk share falls from 20% before the deal to 13% in the year after, and UBS's revenue share from 29% to 17%.

In [ ]:
# 3.6 semantic drift, tested against chance. Embeddings are centred on the pooled mean, so the banking vocabulary
# both banks share cancels out, then averaged per group. Distance = 1 - cosine between two group centroids, and
# every distance is compared with the same measure after shuffling whole utterances between the two groups
# (500 times): only a value above the shuffled 95% band is a real change.
# Three questions: quarter to quarter; each quarter against 2022 (the year before the deals); and each window
# against 2022. The main run uses the chosen model's pre-trained vectors on purpose: drift should catch any change
# in language, not only change along the seven themes. Checks: the other model, the fine-tuned vectors, and the
# earnings calls alone (deal call left out).
N_SHUFFLE = 500
OTHER = next(t for t in MODELS if t != CHOSEN)
VARIANTS = {f"{CHOSEN}, pre-trained": (CHOSEN, qa.index),
            f"{OTHER}, pre-trained": (OTHER, qa.index),
            f"{CHOSEN}, fine-tuned": (f"{CHOSEN}_finetuned", qa.index),
            f"{CHOSEN}, pre-trained, earnings calls only": (CHOSEN, qa.index[qa["call_type"] == "earnings"])}
MAIN = next(iter(VARIANTS))

def unit(X):
    return X / np.linalg.norm(X, axis=-1, keepdims=True)

def utterance_sums(E, g):
    codes, units = pd.factorize(g["turn_id"])
    sums = np.zeros((len(units), E.shape[1]))
    np.add.at(sums, codes, E[g.index.to_numpy()])
    return sums, np.bincount(codes)

def distance_vs_chance(sums_a, n_a, sums_b, n_b, rng):
    # 1 - cosine between the two groups' centroids, and the same after shuffling utterances between the groups
    observed = 1 - unit(sums_a.sum(0) / n_a.sum()) @ unit(sums_b.sum(0) / n_b.sum())
    S, n = np.vstack([sums_a, sums_b]), np.concatenate([n_a, n_b])
    in_a = np.zeros((N_SHUFFLE, len(n)))
    for i in range(N_SHUFFLE):
        in_a[i, rng.permutation(len(n))[:len(n_a)]] = 1
    ca, cb = unit((in_a @ S) / (in_a @ n)[:, None]), unit(((1 - in_a) @ S) / ((1 - in_a) @ n)[:, None])
    null = 1 - (ca * cb).sum(axis=1)
    return observed, null.mean(), np.percentile(null, 97.5)

rng = np.random.default_rng(SEED)
drift_rows = []
for name, (tag, rows_used) in VARIANTS.items():
    used = qa.loc[rows_used]
    E = emb[tag] - emb[tag][used.index].mean(axis=0)
    parts = {(b, q): utterance_sums(E, g) for (b, q), g in used.groupby(["bank", "quarter"])}
    windows = {(b, w): utterance_sums(E, g) for (b, w), g in used.groupby(["bank", "window"])}
    for b in BANKS:
        for i, q in enumerate(QUARTERS):
            tests = {"against 2022": windows[b, "before"]} if q not in WINDOWS["before"] else {}
            if i > 0:
                tests["quarter to quarter"] = parts[b, QUARTERS[i - 1]]
            for measure, ref in tests.items():
                obs, chance, band = distance_vs_chance(*ref, *parts[b, q], rng)
                drift_rows.append({"variant": name, "bank": b, "measure": measure, "period": q,
                                   "observed": obs, "chance": chance, "chance 95%": band})
        for w in WINDOW_NAMES[1:]:
            obs, chance, band = distance_vs_chance(*windows[b, "before"], *windows[b, w], rng)
            drift_rows.append({"variant": name, "bank": b, "measure": "window against 2022", "period": w,
                               "observed": obs, "chance": chance, "chance 95%": band})
drift = pd.DataFrame(drift_rows)
drift["beyond chance"] = drift["observed"] > drift["chance 95%"]
drift["excess"] = drift["observed"] - drift["chance"]
drift.round(4).to_csv(OUT / "drift_bank_quarter.csv", index=False)

d = drift[drift["variant"] == MAIN]
fig, axes = plt.subplots(2, 2, figsize=(12, 6.4), sharey="row")
for col, b in enumerate(BANKS):
    for row, measure in enumerate(["quarter to quarter", "against 2022"]):
        ax = axes[row, col]
        s = d[(d["bank"] == b) & (d["measure"] == measure)].set_index("period").reindex(QUARTERS)
        ax.fill_between(X, 0, s["chance 95%"], color="#c3c2b7", alpha=0.35, lw=0, label="chance (95% band)")
        ax.plot(X, s["observed"], color=BANK_COLOR[b], marker="o", ms=3.5, label="observed")
        hits = s["beyond chance"].eq(True).to_numpy()
        ax.scatter(X[hits], s["observed"].to_numpy()[hits], s=90, facecolors="none", edgecolors="#0b0b0b",
                   lw=1.1, zorder=3)
        time_axis(ax, f"{b}, {measure}", [b])
for ax in axes[:, 0]:
    ax.set_ylabel(f"1 - cosine, centred {CHOSEN}")
axes[0, 0].legend(loc="upper left", fontsize=8)
fig.suptitle("Did the language move more than chance? Circled: beyond the shuffled 95% band",
             x=0.01, ha="left", fontweight="bold")
fig.text(0.01, 0.005, BREAK_NOTE + " · grey band: what random shuffling of answers produces · "
         "bottom row: each quarter against the four 2022 quarters pooled", fontsize=8, color="#52514e")
plt.tight_layout(rect=(0, 0.03, 1, 0.96))
fig.savefig(FIG / "fig_3.6_drift_vs_chance.png", dpi=150, bbox_inches="tight")
plt.show()

print("steps beyond chance, out of those tested")
display(drift.groupby(["bank", "measure", "variant"], sort=False)["beyond chance"]
        .agg(lambda s: f"{s.sum()} of {s.count()}").unstack("variant")[list(VARIANTS)])
wide = drift.pivot_table(index=["bank", "measure", "period"], columns="variant", values="excess")
print("correlation of the excess over chance with the main run:",
      {v: round(wide[MAIN].corr(wide[v]), 2) for v in VARIANTS if v != MAIN})
print("each window against 2022 (main run): distance, chance level and excess")
display(d[d["measure"] == "window against 2022"].set_index(["bank", "period"])
        [["observed", "chance", "chance 95%", "excess", "beyond chance"]].round(4))

Tests whether the language moved more than random shuffling of answers would produce. Compared with 2022, every window is beyond chance at both banks (3 of 3 each; for example UBS's deal quarters at 0.62 against a chance level of 0.21), and quarter to quarter the language moves beyond chance in 7 of 15 steps at UBS and 4 of 15 at JPMorgan. The result holds with MiniLM (8 and 4 of 15), the fine-tuned vectors (7 and 5) and the earnings calls alone (7 and 5), and the excess over chance correlates 0.85 to 0.96 with the main run.

### Summary

Each of the 10,059 sentences is tagged with one of seven fixed themes by a Sentence-BERT tagger fine-tuned on 56 example sentences. Fine-tuning clearly helps mpnet (50 of 56 held-out examples right, from 43) and barely helps MiniLM (45, from 44), so mpnet is the chosen model; on real sentences the effect is modest, and the two models still give the same theme for only 63% of them. The themes show UBS's integration talk rising from 2.8% of its Q&A in 2022 to 12.1% in the deal quarters and 7% to 8% after, and restructuring from 1.6% to 10.9%, while JPMorgan's integration share stays at 2% to 4% and its risk share falls from 20% to 13%. The language moved beyond chance from 2022 to every later window at both banks, and quarter to quarter more often at UBS (7 of 15 steps) than at JPMorgan (4 of 15). Saved under `notebooks/angelo/outputs/2026-09-30/`: embeddings, fine-tuned models, `themes_sentence.csv` and the theme and drift tables.

The tagger asks the same fixed question of every quarter, so the shares can be tracked over time; its limits are the small number of examples and the weak agreement between the two networks.

## Topics with BERTopic

BERTopic groups sentences that mean similar things and names each group by the words that set it apart from the others (class-based TF-IDF). Nobody tells it which subjects exist, so it shows the agenda as the calls set it. Each sentence becomes a vector (MiniLM, from 3.1); UMAP squeezes the vectors into five dimensions; HDBSCAN finds dense groups, and sentences that fit no group stay unassigned. One model is fitted on both banks and all 16 quarters, so a topic means the same thing for UBS and JPMorgan in every quarter. Topics are labelled automatically by their top words; no topic is named or edited by hand.

**Tuning.** The setting that matters most is HDBSCAN's minimum cluster size, the smallest group that counts as a topic. It is tried at 15, 25, 40, 60, 80, 100, 150 and 200 sentences, each with three UMAP seeds, and every setting is scored on three properties:

- **Stability:** do the same topics come back with another random start? Adjusted Rand index (ARI) between the seed-42 run and the two other seeds, on the sentences both runs assign.
- **Coherence:** do a topic's top 10 words actually appear together? Normalised pointwise mutual information (NPMI) of every pair of top words, counted over speaker turns, from -1 (never together) to 1 (always together), averaged over topics. A single sentence is too short a context for ten words, so the speaker turn is the unit, as in the standard coherence measures that count co-occurrence within a window or document (Röder, Both and Hinneburg, 2015). The sentence-level value is reported as a check.
- **Diversity:** the share of distinct words among all topics' top words; 1 means no two topics share a top word.

**Rule for the final setting, fixed before the run:** the highest topic quality, coherence times diversity (Dieng, Ruiz and Blei, 2020), among the settings whose topics are stable across seeds (mean ARI of at least 0.80, "good" agreement in Steinley's 2004 scale). The rule replaces a choice by eye with published measures.

**Merger-related topics, rule fixed before the run:** a topic is merger-related when the share of its sentences with a merger keyword (2.4) is at least four times the corpus rate and at least 10 of its sentences have one.

In [ ]:
# 4.1 stop list for the topic labels only (CountVectorizer), never applied to the text the models read
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

name_tokens = {t for n in utt_raw["speaker_name"].dropna().unique() for t in re.findall(r"[a-z]{2,}", n.lower())}
other_names = {  # short forms, and executives named in the Q&A who never speak
    "steve", "ben", "nick", "andy", "charlie", "ralph", "colm", "kelleher", "iqbal", "khan",
    "marianne", "jen", "jennifer", "doug", "mary", "daniel", "pinto", "sergio", "todd", "jamie", "jeremy", "mikael"}
filler_words = {  # contraction fragments and spoken fillers, plus time words
    "ve", "ll", "re", "don", "didn", "doesn", "isn", "wasn", "aren", "won", "wouldn", "couldn", "haven", "hasn",
    "just", "thank", "thanks", "okay", "ok", "right", "good", "great", "morning", "yeah", "yes", "sure",
    "know", "think", "maybe", "wanted", "guess", "really", "actually", "obviously", "look", "mean",
    "kind", "sort", "bit", "little", "lot", "going", "terms", "thing", "things", "way", "say", "said",
    "got", "like", "want", "talk", "talked", "talking", "question", "questions", "helpful", "color", "colour",
    "quarter", "quarters", "year", "years", "today",
    # BERTopic strips apostrophes before counting words, so contractions arrive as one word
    "thats", "dont", "im", "ill", "id", "youre", "theyre", "weve", "whats", "doesnt", "didnt", "isnt", "wasnt",
    "shouldnt", "wont", "cant", "ive", "youve", "lets", "theres", "hes", "wouldnt", "couldnt", "arent", "havent",
    "hasnt", "et", "cetera", "guys"}
NAMES = name_tokens | other_names
NAMES |= {n + "s" for n in NAMES}                                   # possessives: "Jamie's" arrives as "jamies"
STOP_WORDS = sorted(ENGLISH_STOP_WORDS | NAMES | filler_words)
pd.Series(STOP_WORDS, name="token").to_csv(OUT / "stoplist_topic_labels.csv", index=False)
print(f"{len(STOP_WORDS)} stop words: {len(ENGLISH_STOP_WORDS)} English, {len(name_tokens)} speaker-name tokens, "
      f"{len(other_names)} other names, {len(filler_words)} fillers")

Builds the word list used only to name the topics, never to change the text the models read: 318 English stop words, 91 speaker-name tokens, 22 other names and 93 fillers and contraction fragments, 623 in all. This keeps first names and spoken fillers out of the topic labels; the tuning table in 4.2 confirms that no top-10 list contains a name.

In [ ]:
# 4.2 tuning: eight minimum cluster sizes, each with three UMAP seeds. UMAP runs once per seed; BERTopic then
# clusters the reduced vectors, so all settings of one seed see the same reduction. Seed 42 is the full BERTopic
# fit (topics, words); seeds 43 and 44 only check whether the groups come back. Coherence counts word pairs with
# BERTopic's own text cleaning and vocabulary, over speaker turns (main) and over single sentences (check).
from bertopic import BERTopic
from bertopic.dimensionality import BaseDimensionalityReduction
from umap import UMAP
from hdbscan import HDBSCAN
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import adjusted_rand_score

GRID = [15, 25, 40, 60, 80, 100, 150, 200]
MIN_ARI = 0.80                                     # "good" agreement (Steinley 2004)
SEEDS = [SEED, SEED + 1, SEED + 2]
UMAP_ARGS = {"n_neighbors": 15, "n_components": 5, "min_dist": 0.0, "metric": "cosine"}
docs = qa["text"].tolist()

t0 = time.time()
reduced = {s: UMAP(random_state=s, **UMAP_ARGS).fit_transform(emb["MiniLM"]) for s in SEEDS}
print(f"UMAP for {len(SEEDS)} seeds in {time.time() - t0:.0f}s")

def vectorizer():
    return CountVectorizer(stop_words=STOP_WORDS, ngram_range=(1, 2), min_df=5)

def clusterer(m):
    return HDBSCAN(min_cluster_size=m, min_samples=5, metric="euclidean", cluster_selection_method="eom",
                   prediction_data=True)

def fit_bertopic(m, seed=SEED):
    model = BERTopic(umap_model=BaseDimensionalityReduction(), hdbscan_model=clusterer(m),
                     vectorizer_model=vectorizer(), top_n_words=10, calculate_probabilities=False)
    topics, _ = model.fit_transform(docs, embeddings=reduced[seed])
    return model, np.asarray(topics)

def topic_words(model, n=10):
    # BERTopic pads a topic with blank words when it has fewer than n
    return {t: [w for w, _ in model.get_topic(t) if w][:n] for t in sorted(set(model.topics_)) if t != -1}

def agreement(a, b):
    # adjusted Rand index on the sentences both runs assign to a topic
    ok = (a != -1) & (b != -1)
    return adjusted_rand_score(a[ok], b[ok])

# word presence per sentence and per speaker turn, as BERTopic counts words (letters and digits only)
clean_docs = [re.sub(r"[^A-Za-z0-9 ]+", "", d.replace("\n", " ").replace("\t", " ")) for d in docs]
vocab_vec = vectorizer().fit(clean_docs)
VOCAB = vocab_vec.vocabulary_
in_sentence = (vocab_vec.transform(clean_docs) > 0).astype(np.int32)
turn_codes, turn_ids = pd.factorize(qa["turn_id"])
to_turn = sparse.csr_matrix((np.ones(len(turn_codes)), (turn_codes, np.arange(len(turn_codes)))),
                            shape=(len(turn_ids), len(turn_codes)))
in_turn = ((to_turn @ in_sentence) > 0).astype(np.int32)

def npmi(words, X):
    # mean NPMI over all pairs of the words; a pair that never appears together scores -1
    cols = X[:, [VOCAB[w] for w in words]].tocsc()
    p = np.asarray(cols.sum(axis=0)).ravel() / X.shape[0]
    co = (cols.T @ cols).toarray() / X.shape[0]
    pairs = [(i, j) for i in range(len(words)) for j in range(i + 1, len(words))]
    return np.mean([-1.0 if co[i, j] == 0 else np.log(co[i, j] / (p[i] * p[j])) / -np.log(co[i, j])
                    for i, j in pairs])

fits, tuning = {}, []
t0 = time.time()
for m in GRID:
    fits[m] = fit_bertopic(m)
    labels = {SEED: fits[m][1], **{s: clusterer(m).fit_predict(reduced[s]) for s in SEEDS[1:]}}
    words = topic_words(fits[m][0])
    sizes = pd.Series(labels[SEED][labels[SEED] != -1]).value_counts()
    diversity = len({w for ws in words.values() for w in ws}) / sum(len(ws) for ws in words.values())
    coherence = np.mean([npmi(ws, in_turn) for ws in words.values()])
    coherence_sent = np.mean([npmi(ws, in_sentence) for ws in words.values()])
    tuning.append({
        "min_cluster_size": m, "topics": len(words),
        "topics, other seeds": " / ".join(str(len(set(labels[s]) - {-1})) for s in SEEDS[1:]),
        "unassigned %": 100 * (labels[SEED] == -1).mean(),
        "largest topic %": 100 * sizes.max() / len(docs),
        "agreement across seeds (ARI)": np.mean([agreement(labels[SEED], labels[s]) for s in SEEDS[1:]]),
        "coherence (NPMI, turns)": coherence, "diversity": diversity,
        "topic quality": coherence * diversity,
        "coherence (NPMI, sentences)": coherence_sent, "topic quality, sentence context": coherence_sent * diversity,
        "name words in top-10": sum(any(p in NAMES for p in w.split()) for ws in words.values() for w in ws),
    })
print(f"{len(GRID)} settings x {len(SEEDS)} seeds clustered and scored in {time.time() - t0:.0f}s")
tuning = pd.DataFrame(tuning).set_index("min_cluster_size")

stable = tuning[tuning["agreement across seeds (ARI)"] >= MIN_ARI]
MIN_TOPIC = stable["topic quality"].idxmax()
display(tuning.round(3))
print(f"stable settings (ARI >= {MIN_ARI}): {stable.index.tolist()}; chosen minimum cluster size: {MIN_TOPIC}; "
      f"with sentences as the coherence context the rule would choose {stable['topic quality, sentence context'].idxmax()}")

runs = pd.DataFrame([{"run": f"minimum {m}" + (" (chosen)" if m == MIN_TOPIC else ""), "unit": "sentence",
                      "data": "2022 to 2025", "topics": str(r["topics"]), "unassigned %": f"{r['unassigned %']:.1f}"}
                     for m, r in tuning.iterrows()]).set_index("run")
display(runs)
tuning.round(4).to_csv(OUT / "topic_tuning.csv")
runs.to_csv(OUT / "topic_runs.csv")

panels = [("topics", "Topics found", "{:.0f}"), ("unassigned %", "Sentences left unassigned, %", "{:.0f}"),
          ("agreement across seeds (ARI)", "Same topics with another seed (ARI)", "{:.2f}"),
          ("topic quality", "Topic quality (coherence x diversity)", "{:.3f}")]
fig, axes = plt.subplots(1, 4, figsize=(15, 3))
gx = np.arange(len(GRID))
axes[2].axhline(MIN_ARI, color="#898781", lw=1, ls="--")
axes[2].text(len(GRID) - 1, MIN_ARI, "stability threshold ", ha="right", va="bottom", fontsize=7, color="#898781")
for ax, (col, title, fmt) in zip(axes, panels):
    ax.plot(gx, tuning[col], color="#4a3aa7", marker="o")
    ax.scatter([GRID.index(MIN_TOPIC)], [tuning.loc[MIN_TOPIC, col]], s=120, facecolors="none", edgecolors="#0b0b0b",
               zorder=3)
    for i, m in enumerate(GRID):
        ax.annotate(fmt.format(tuning.loc[m, col]), (i, tuning.loc[m, col]), xytext=(0, 7), textcoords="offset points",
                    ha="center", fontsize=7)
    ax.set_xticks(gx, GRID)
    ax.set_xlabel("HDBSCAN minimum cluster size")
    ax.set_title(title, loc="left")
    ax.margins(y=0.25)
fig.suptitle("BERTopic tuning on 2022 to 2025 Q&A sentences (circled: chosen by the rule)", x=0.01, ha="left",
             fontweight="bold")
plt.tight_layout(rect=(0, 0, 1, 0.92))
fig.savefig(FIG / "fig_4.2_topic_tuning.png", dpi=150, bbox_inches="tight")
plt.show()

Fits BERTopic at eight minimum cluster sizes with three random starts each and scores them on stability, coherence and diversity. Larger sizes score higher on topic quality, but the largest (200) is not stable across starts (agreement 0.45 against the 0.80 required), so the rule picks 150: 17 topics, 40.5% of sentences unassigned and agreement 0.93. At 100 the model finds 25 topics with 36.5% unassigned, and no top-10 list contains a name at any size.

In [ ]:
# 4.3 the final model and its topics. Each topic is labelled automatically by its top words (a word already inside
# an earlier two-word term is skipped) and described by its main theme from the tagger (3.4). Coherence per topic
# as in 4.2. A topic is merger-related by the rule fixed above.
topic_model, topics = fits[MIN_TOPIC]
qa["topic"] = topics
words = topic_words(topic_model)

def short_label(ws, n=3):
    chosen, seen = [], set()
    for w in ws:
        if set(w.split()) <= seen:
            continue
        chosen.append(w)
        seen |= set(w.split())
        if len(chosen) == n:
            break
    return ", ".join(chosen)

def topic_label(t):
    return "unassigned" if t == -1 else f"{t}: {short_label(words[t])}"

counts = pd.crosstab(qa["topic"], qa["bank"])
share_bank = 100 * counts / counts.sum()
theme_mix = pd.crosstab(qa["topic"], qa["theme"], normalize="index").reindex(columns=THEMES, fill_value=0)
MERGER_RATE = qa["merger_keyword"].mean()
kw = qa.groupby("topic")["merger_keyword"].agg(["mean", "sum"]).drop(index=-1)
MERGER_TOPICS = kw.index[(kw["mean"] >= 4 * MERGER_RATE) & (kw["sum"] >= 10)].tolist()
topic_table = pd.DataFrame({
    "label": pd.Series({t: topic_label(t) for t in words}),
    "sentences": counts.sum(axis=1),
    "UBS %": share_bank["UBS"], "JPM %": share_bank["JPM"],
    "analyst %": 100 * pd.crosstab(qa["topic"], qa["speaker_role"], normalize="index")["analyst"],
    "main theme": theme_mix.idxmax(axis=1), "main theme %": 100 * theme_mix.max(axis=1),
    "merger keyword %": 100 * kw["mean"],
    "coherence": pd.Series({t: npmi(ws, in_turn) for t, ws in words.items()}),
    "top words": pd.Series({t: ", ".join(ws[:7]) for t, ws in words.items()}),
}).drop(index=-1).sort_values("sentences", ascending=False)
topic_table["merger-related"] = topic_table.index.isin(MERGER_TOPICS)
qa["topic_label"] = qa["topic"].map(topic_label)
topic_table.round(3).to_csv(OUT / "topic_info.csv")
qa[["key", "sentence_id", "topic", "topic_label"]].to_csv(OUT / "topics_sentence.csv", index=False)
print(f"{len(topic_table)} topics; unassigned {100 * (qa['topic'] == -1).mean():.1f}% of sentences; corpus "
      f"merger-keyword rate {100 * MERGER_RATE:.1f}%; merger-related topics: {[topic_label(t) for t in MERGER_TOPICS]}")
display(topic_table.round(2))

Fits the chosen model: 17 topics cover 59.5% of the sentences and 40.5% stay unassigned. The largest topic (1,763 sentences, 17.5%) is spoken framing with no subject of its own, and the others are business subjects such as costs, lending, deposits, capital and rates, and net interest income (NII), whose label "nii, gwm, income" also carries GWM, UBS's Global Wealth Management division. One topic is merger-related by the rule fixed in advance, Credit Suisse and Switzerland (221 sentences, 45% with a merger keyword), which is 5.0% of UBS's sentences and 0.1% of JPMorgan's.

In [ ]:
# 4.4 three sentences nearest each topic's centre (MiniLM vectors), printed so a reader can check the automatic
# labels; nothing in the analysis uses them. Prints transcript text: clear outputs before committing.
E = emb["MiniLM"]
topic_arr = qa["topic"].to_numpy()
for t in topic_table.index:
    idx = np.flatnonzero(topic_arr == t)
    nearest = idx[np.argsort(-(E[idx] @ E[idx].mean(axis=0)))[:3]]
    print(f"\n{topic_label(t)} · {len(idx)} sentences · main theme {topic_table.at[t, 'main theme']}")
    for i in nearest:
        print(f"   {qa.at[i, 'bank']} {qa.at[i, 'quarter']} {qa.at[i, 'speaker_role'][:4]}  {qa.at[i, 'text'][:160]}")

Prints the three sentences nearest each topic's centre so a reader can check the automatic labels; they match the subject of each topic, for example cost saves, net new assets in Asia and the integration of Credit Suisse. The largest topic holds sentences of spoken framing with no subject of their own, which is why the tagger gives it the main theme "other".

In [ ]:
# 4.5 topic shares per bank and window, with 95% intervals from resampling whole utterances. Shares are of all the
# bank's sentences in the window, unassigned ones included, so they compare across banks and windows. Then the
# change from the year before the deals (2022) to the year after (2023-Q3 to 2024-Q2), with a 95% interval.
TOPICS = [-1] + topic_table.index.tolist()

def before_after(df, col, cats, rng):
    # share of each category before and after per bank, the change, and a 95% interval for the change
    rows = []
    for b in BANKS:
        point, boot = {}, {}
        for w in ("before", "after"):
            point[w], boot[w] = shares(df[(df["bank"] == b) & (df["window"] == w)], col, cats, rng)
        lo, hi = np.percentile(100 * (boot["after"] - boot["before"]), [2.5, 97.5], axis=0)
        rows += [{"bank": b, "topic": c, "before %": 100 * point["before"][k], "after %": 100 * point["after"][k],
                  "lo": lo[k], "hi": hi[k]} for k, c in enumerate(cats)]
    out = pd.DataFrame(rows)
    out["change"] = out["after %"] - out["before %"]
    out["clear"] = (out["lo"] > 0) | (out["hi"] < 0)
    return out

rng = np.random.default_rng(SEED)
rows = []
for (b, w), g in qa.groupby(["bank", "window"]):
    point, boot = shares(g, "topic", TOPICS, rng)
    lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
    rows += [{"bank": b, "window": w, "topic": t, "label": topic_label(t), "share": 100 * point[k],
              "lo": 100 * lo[k], "hi": 100 * hi[k]} for k, t in enumerate(TOPICS)]
topic_win = pd.DataFrame(rows)
topic_win.round(2).to_csv(OUT / "topic_shares_bank_window.csv", index=False)

show = topic_table.index.tolist()
mat = topic_win.pivot_table(index="topic", columns=["bank", "window"], values="share").reindex(
    index=show, columns=pd.MultiIndex.from_product([BANKS, WINDOW_NAMES]))
fig, ax = plt.subplots(figsize=(9.5, 0.3 * len(show) + 1.8))
im = ax.imshow(mat.to_numpy(), aspect="auto", cmap=BLUES, vmin=0)
for (i, j), v in np.ndenumerate(mat.to_numpy()):
    ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=7,
            color="white" if v > 0.6 * np.nanmax(mat.to_numpy()) else "#0b0b0b")
ax.set_xticks(range(mat.shape[1]), [f"{b}\n{w}" for b, w in mat.columns], fontsize=8)
ax.set_yticks(range(len(show)), [topic_label(t) + (" *" if t in MERGER_TOPICS else "") for t in show], fontsize=8)
ax.axvline(len(WINDOW_NAMES) - 0.5, color="#52514e", lw=1.5)
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.6, label="% of the bank's Q&A sentences in the window")
ax.set_title(f"The {len(show)} topics by bank and window (* merger-related)", loc="left")
plt.tight_layout()
fig.savefig(FIG / "fig_4.5_topics_by_window.png", dpi=150, bbox_inches="tight")
plt.show()

moves = before_after(qa, "topic", TOPICS, np.random.default_rng(SEED))
moves.insert(2, "label", moves["topic"].map(topic_label))
moves.round(2).to_csv(OUT / "topic_change_before_after.csv", index=False)
print("changes from before (2022) to after (2023-Q3 to 2024-Q2) whose 95% interval excludes zero, largest first")
for b in BANKS:
    m = moves[(moves["bank"] == b) & moves["clear"]]
    display(m.reindex(m["change"].abs().sort_values(ascending=False).index).round(1))

Compares the year before the deals (2022) with the year after (2023-Q3 to 2024-Q2). At UBS six topics change beyond their ranges: Credit Suisse and Switzerland (1.8% to 5.6%), the capital topics (RWA and CET1 from 1.1% to 4.7%, excess capital from 0.7% to 3.3%) and spoken framing (12% to 16%) rise, while wealth management (10% to 3%) and net new assets in Asia (8% to 2%) fall. At JPMorgan only one small topic moves.

In [ ]:
# 4.6 merger-related topics per quarter: their combined share of each bank's sentences, with 95% intervals
qa["merger_topic"] = qa["topic"].isin(MERGER_TOPICS)
rng = np.random.default_rng(SEED)
rows = []
for (b, q), g in qa[qa["call_type"] == "earnings"].groupby(["bank", "quarter"]):
    point, boot = shares(g, "merger_topic", [False, True], rng)
    lo, hi = np.percentile(boot[:, 1], [2.5, 97.5])
    rows.append({"bank": b, "quarter": q, "share": 100 * point[1], "lo": 100 * lo, "hi": 100 * hi})
merger_q = pd.DataFrame(rows).set_index(["bank", "quarter"])
deal_call = 100 * qa.loc[qa["call_type"] == "event", "merger_topic"].mean()

fig, ax = plt.subplots(figsize=(9.5, 3.6))
for b in BANKS:
    s = merger_q.xs(b, level="bank").reindex(QUARTERS)
    ax.fill_between(X, s["lo"], s["hi"], color=BANK_COLOR[b], alpha=0.12, lw=0)
    ax.plot(X, s["share"], color=BANK_COLOR[b], marker="o", ms=3.5, label=b)
mark_event(ax, deal_call)
time_axis(ax, "Merger-related topics, % of each bank's Q&A sentences")
ax.set_ylim(bottom=0)
ax.legend(loc="upper right", fontsize=8)
fig.text(0.01, 0.005, BREAK_NOTE + " · shaded bands: 95% interval from resampling answers · " + EVENT_NOTE,
         fontsize=7, color="#52514e")
plt.tight_layout(rect=(0, 0.04, 1, 1))
fig.savefig(FIG / "fig_4.6_merger_topics.png", dpi=150, bbox_inches="tight")
plt.show()
merger_q.round(2).to_csv(OUT / "merger_topics_bank_quarter.csv")
display(merger_q["share"].unstack("quarter").reindex(columns=QUARTERS).round(1))
print(f"JPM deal call alone: {deal_call:.1f}%")

Follows the Credit Suisse topic quarter by quarter. It is 1.4% to 2.6% of UBS's Q&A in 2022, 8.9% and 11.4% in 2023-Q1 and Q2, about 5% to 2024-Q3, 2% to 3% in the next two quarters and 13.3% in 2025-Q3. At JPMorgan the share is close to zero in every quarter, and the First Republic deal call alone reaches 0.6%, so a topic model of this size does not see JPMorgan's merger talk; the theme tagger and the keyword search do.

In [ ]:
# 4.7 do the results depend on the unassigned sentences? Each unassigned sentence is given the topic whose centre
# (mean MiniLM vector of its sentences) is closest by cosine, and the before-and-after changes of 4.5 are computed
# again. A change that keeps its direction and stays beyond its range does not depend on the unassigned sentences.
E = emb["MiniLM"]
T = qa["topic"].to_numpy()
ids = [t for t in TOPICS if t != -1]
centres = unit(np.vstack([E[T == t].mean(axis=0) for t in ids]))
qa["topic_all"] = T
qa.loc[T == -1, "topic_all"] = np.array(ids)[(E[T == -1] @ centres.T).argmax(axis=1)]
print(f"{(T == -1).sum():,} unassigned sentences reassigned; the largest topic now holds "
      f"{100 * qa['topic_all'].value_counts(normalize=True).max():.1f}% of sentences")

moves_all = before_after(qa, "topic_all", ids, np.random.default_rng(SEED))
check = moves[moves["topic"] != -1].merge(moves_all, on=["bank", "topic"], suffixes=("", ", all assigned"))
check["same direction"] = np.sign(check["change"]) == np.sign(check["change, all assigned"])
main_clear = check[check["clear"]]
print(f"changes beyond their range in 4.5: {len(main_clear)}; with every sentence assigned they keep their direction "
      f"in {int(main_clear['same direction'].sum())} and stay beyond their range in "
      f"{int((main_clear['same direction'] & main_clear['clear, all assigned']).sum())}")
display(check[check["clear"] | check["clear, all assigned"]]
        [["bank", "label", "change", "lo", "hi", "change, all assigned", "lo, all assigned", "hi, all assigned",
          "clear, all assigned"]].round(1))

Gives each of the 4,075 unassigned sentences the nearest topic and recomputes the changes of 4.5. All 7 changes beyond their range keep their direction and stay beyond it, so the results do not depend on the unassigned sentences.

In [ ]:
# 4.8 the two models side by side: how the topics spread over the themes. Agreement is measured with adjusted mutual
# information (0 = no more than chance, 1 = the same grouping) on assigned sentences.
from sklearn.metrics import adjusted_mutual_info_score

assigned = qa[qa["topic"] != -1]
ct = pd.crosstab(assigned["topic"], assigned["theme"]).reindex(columns=THEMES, fill_value=0)
ct.to_csv(OUT / "topic_theme_crosstab.csv")
ami = adjusted_mutual_info_score(assigned["topic"], assigned["theme"])
print(f"adjusted mutual information, topics against themes: {ami:.2f}")
print(f"sentences whose theme is their topic's main theme: {100 * ct.max(axis=1).sum() / ct.to_numpy().sum():.1f}%")
print("main theme of the topics:", topic_table["main theme"].value_counts().reindex(THEMES, fill_value=0).to_dict())
print("unassigned sentences by theme (%):",
      (100 * qa.loc[qa["topic"] == -1, "theme"].value_counts(normalize=True)).reindex(THEMES).round(1).to_dict())
print("merger-related topics: share of their sentences the tagger calls integration (%)",
      {topic_label(t): round(100 * (qa.loc[qa["topic"] == t, "theme"] == "integration").mean(), 1) for t in MERGER_TOPICS})

rown = 100 * ct.loc[show].div(ct.loc[show].sum(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(8.5, 0.3 * len(show) + 1.6))
im = ax.imshow(rown.to_numpy(), aspect="auto", cmap=BLUES, vmin=0, vmax=100)
for (i, j), v in np.ndenumerate(rown.to_numpy()):
    if v >= 10:
        ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=7, color="white" if v > 60 else "#0b0b0b")
ax.set_xticks(range(len(THEMES)), THEMES, fontsize=8)
ax.set_yticks(range(len(show)), [topic_label(t) for t in show], fontsize=8)
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.6, label="% of the topic's sentences")
ax.set_title(f"BERTopic topics against Sentence-BERT themes (AMI {ami:.2f})", loc="left")
plt.tight_layout()
fig.savefig(FIG / "fig_4.8_topics_vs_themes.png", dpi=150, bbox_inches="tight")
plt.show()

Compares the two models: topics and themes agree only modestly (adjusted mutual information 0.25; 61% of sentences carry their topic's main theme). Eight topics are mainly revenue, four capital, two other, and one each risk, costs and integration. The Credit Suisse topic is mainly integration, yet only 27% of its sentences are tagged integration, so the two methods describe the corpus differently rather than repeat each other.

In [ ]:
# 4.9 how much does the topic model depend on tiny differences in the text? About one sentence in ten contains curly
# quotes or long dashes. Here they are written with plain characters (the vectors barely move, see the first table) and
# BERTopic is refitted at every minimum cluster size with the same seed and settings. Nothing else changes.
plain_idx = qa.index[qa["typography_changed"]]
plain_model = SentenceTransformer(MODELS["MiniLM"], device=DEVICE)
E_plain = emb["MiniLM"].copy()
E_plain[plain_idx] = encode(plain_model, qa.loc[plain_idx, "text_plain"], 64)
del plain_model
free_gpu()
cos = (E_plain[plain_idx] * emb["MiniLM"][plain_idx]).sum(axis=1)
typo = (pd.DataFrame({"bank": qa.loc[plain_idx, "bank"].to_numpy(), "cosine": cos})
        .groupby("bank")["cosine"].agg(sentences="size", mean="mean", lowest="min"))
typo["share of the bank's corpus %"] = 100 * typo["sentences"] / qa["bank"].value_counts()
typo.round(4).to_csv(OUT / "typography_effect_minilm.csv")
display(typo.round(4))

red_plain = UMAP(random_state=SEED, **UMAP_ARGS).fit_transform(E_plain)
docs_plain = qa["text_plain"].tolist()
rows = []
for m in GRID:
    model_p = BERTopic(umap_model=BaseDimensionalityReduction(), hdbscan_model=clusterer(m),
                       vectorizer_model=vectorizer(), top_n_words=10, calculate_probabilities=False)
    t_p = np.asarray(model_p.fit_transform(docs_plain, embeddings=red_plain)[0])
    t_a = fits[m][1]
    both = (t_a != -1) & (t_p != -1)
    row = {"min_cluster_size": m}
    for name, t in (("as spoken", t_a), ("plain characters", t_p)):
        kw_t = pd.DataFrame({"t": t, "k": qa["merger_keyword"]}).groupby("t")["k"].agg(["mean", "sum"]).drop(index=-1, errors="ignore")
        mt = kw_t.index[(kw_t["mean"] >= 4 * MERGER_RATE) & (kw_t["sum"] >= 10)]
        sizes = pd.Series(t).value_counts()
        row[f"topics, {name}"] = len(set(t) - {-1})
        row[f"merger-related topic sizes, {name}"] = sorted((int(sizes[x]) for x in mt), reverse=True)
    row["same topics (ARI, shared assigned)"] = adjusted_rand_score(t_a[both], t_p[both])
    rows.append(row)
robust = pd.DataFrame(rows).set_index("min_cluster_size")
robust.round(3).to_csv(OUT / "topic_text_robustness.csv")
display(robust.round(2))


Tests how sensitive the topic model is to tiny text changes. About one sentence in ten has curly quotes or long dashes (9.7% at JPMorgan, 11.0% at UBS), and writing them as plain characters moves their vectors by only 0.5% (mean cosine 0.995), yet the refitted model differs: the Credit Suisse topic has 172 sentences instead of 221 at minimum 100 and 150 and disappears at 200, and the groupings agree only 0.79 to 0.92. The topic model is therefore sensitive to small changes in its input, which is why no extra sentences are deleted from the corpus and why the Credit Suisse topic is read together with the theme tagger and the keyword search.

### Summary

BERTopic, with its minimum cluster size chosen by a rule fixed in advance, finds 17 topics for 59.5% of the sentences; 40.5% stay unassigned. The largest topic is spoken framing with no subject of its own (1,763 sentences); the others are costs, NII and income, lending, deposits, capital, rates, wealth management and net new assets, and one merger-related topic, Credit Suisse and Switzerland (221 sentences, 5.0% of UBS's Q&A against 0.1% of JPMorgan's). From the year before the deals to the year after, UBS's Credit Suisse topic rises from 1.8% to 5.6% and its capital topics rise, while wealth management and net new assets in Asia fall; only one small JPMorgan topic moves. These changes keep their direction when every unassigned sentence is given a topic, but the model is sensitive to small text changes: with plain quotes and dashes the Credit Suisse topic shrinks from 221 to 172 sentences at the chosen setting and disappears at the largest.

Topics show the agenda as the calls set it, and the Credit Suisse topic follows the same pattern as the integration theme: low in 2022 and highest in the deal quarters. A JPMorgan merger topic is too small for a topic model of this size, so its merger talk is read through the theme tagger and the keyword search.

## Comparison with the sentiment models

The team's sentiment analysis ran four models (FinBERT, FinBERT-tone, RoBERTa and an emotion model) on the same release and flagged six quarters where the tone changed. None of those changes was confirmed by more than two models at once. This section asks whether the language itself also moved in those quarters: does the topic mix, the theme mix or the overall meaning (drift, 3.6) change more than chance from the quarter before? The same tests run on every other quarter, so the flagged quarters can be compared with a base rate.

This compares at quarter level. The next section goes down to single sentences: it joins the sentiment labels on the key (bank, quarter, call type, position in call, sentence number) to find the topics within negative sentences.

In [ ]:
# 5.1 quarter-to-quarter shift in the topic mix and the theme mix, tested against chance like the drift in 3.6:
# total variation distance (half the sum of absolute share differences) between two quarters, against the same
# measure after shuffling whole utterances between them 500 times. Then the six flagged quarters against the rest.
# the six quarters named in the sentiment group's report ("Sentiment Classification"); update if their list changes
FLAGGED = {("UBS", "2022-Q4"): "FinBERT-tone, prepared remarks",
           ("UBS", "2024-Q1"): "RoBERTa, more negative Q&A",
           ("UBS", "2025-Q4"): "RoBERTa, more negative Q&A",
           ("JPM", "2024-Q3"): "RoBERTa, more negative Q&A",
           ("JPM", "2025-Q1"): "RoBERTa, more negative Q&A",
           ("JPM", "2025-Q2"): "RoBERTa, more negative Q&A"}

def mix_shift_vs_chance(counts_a, counts_b, rng):
    tvd = lambda a, b: 0.5 * np.abs(a / a.sum(-1, keepdims=True) - b / b.sum(-1, keepdims=True)).sum(-1)
    observed = tvd(counts_a.sum(0), counts_b.sum(0))
    S = np.vstack([counts_a, counts_b])
    in_a = np.zeros((N_SHUFFLE, len(S)))
    for i in range(N_SHUFFLE):
        in_a[i, rng.permutation(len(S))[:len(counts_a)]] = 1
    null = tvd(in_a @ S, (1 - in_a) @ S)
    return observed, null.mean(), np.percentile(null, 97.5)

rng = np.random.default_rng(SEED)
shift_rows = []
for b in BANKS:
    for i in range(1, len(QUARTERS)):
        prev, q = QUARTERS[i - 1], QUARTERS[i]
        ga, gb = qa[(qa["bank"] == b) & (qa["quarter"] == prev)], qa[(qa["bank"] == b) & (qa["quarter"] == q)]
        row = {"bank": b, "quarter": q, "flagged": FLAGGED.get((b, q), "")}
        for col, cats in (("topic", TOPICS), ("theme", THEMES)):
            obs, chance, band = mix_shift_vs_chance(unit_counts(ga, col, cats), unit_counts(gb, col, cats), rng)
            row[f"{col} shift"], row[f"{col} beyond chance"] = obs, obs > band
            pa, pb = (100 * g[col].value_counts(normalize=True) for g in (ga, gb))
            diff = pb.reindex(cats, fill_value=0) - pa.reindex(cats, fill_value=0)
            top_move = diff.abs().idxmax()
            row[f"largest {col} move"] = (f"{topic_label(top_move) if col == 'topic' else top_move} "
                                          f"{diff[top_move]:+.1f} pts")
        dd = drift[(drift["variant"] == MAIN) & (drift["bank"] == b) & (drift["measure"] == "quarter to quarter")
                   & (drift["period"] == q)].iloc[0]
        row["drift beyond chance"] = bool(dd["beyond chance"])
        shift_rows.append(row)
shift = pd.DataFrame(shift_rows)
shift.round(4).to_csv(OUT / "sentiment_quarters_check.csv", index=False)

print("the six quarters flagged by the sentiment models")
display(shift[shift["flagged"] != ""].set_index(["bank", "quarter"])[
    ["flagged", "drift beyond chance", "topic beyond chance", "theme beyond chance",
     "largest topic move", "largest theme move"]])
tests = ["drift beyond chance", "topic beyond chance", "theme beyond chance"]
base_rate = shift.assign(group=np.where(shift["flagged"] != "", "flagged quarters", "other quarters")) \
            .groupby("group")[tests].agg(lambda s: f"{s.sum()} of {s.count()}")
print("how often the language moved beyond chance: flagged quarters against the rest")
display(base_rate)

Checks the six quarters the sentiment models flagged. In half of them the language moved beyond chance from the quarter before (3 of 6, against 8 of the 24 other quarters) and the theme mix shifted beyond chance in 3 of 6 (4 of 24 otherwise), but the topic mix did not in any of them (4 of 24 otherwise). Six quarters are too few to prove a link, but tone, language and themes move together in UBS 2025-Q4 (Credit Suisse talk down almost 10 points) and JPMorgan 2025-Q2 (more capital talk).

### Summary

The six quarters where the sentiment models saw a tone change were tested for a change in what was said. Three of the six also show a change in language beyond chance (as do 8 of the 24 other quarters), the theme mix shifts beyond chance in three of six (4 of 24), and the topic mix in none (4 of 24). The clearest matches are UBS 2025-Q4, where language and themes move as Credit Suisse talk drops by almost 10 points and revenue talk rises, and JPMorgan 2025-Q2, with more capital talk; one flag (UBS 2022-Q4) comes from the prepared remarks, which this notebook does not model.

Six quarters are too few for a firm conclusion. The comparison gives the sentiment results a subject: when tone and theme change together, the theme says what the tone is about. The next section joins the sentiment labels sentence by sentence to show which topics carry the negative tone.

## Topics within negative sentences

The sentiment track labels every sentence with three sentiment models (FinBERT, FinBERT-tone and RoBERTa). Its two output files, one per bank, sit in the data folder with the sentence key, and this section joins them to the topics and themes (the files also hold an emotion model, which is not used here). If the files change, only `LABEL_FILES` and `LABEL_DEFS` at the top of 6.1 need editing.

**Definition.** A sentence is negative when FinBERT calls it negative: FinBERT is the model trained on financial text and the one in the project's scope. A strict version counts a sentence as negative only when at least two of the three sentiment models agree. Two questions per bank, comparing the year before the deals (2022) with the year after (2023-Q3 to 2024-Q2): which topics do the negative sentences talk about, and which topics are most often negative?

**Limit.** The sentiment track's models read cleaned text, from which negations, numbers and many direction words are removed (for example "we don't expect a loss" reaches the model as "expect loss"). The labels used here carry that limit; they are the sentiment track's results, not a new sentiment model.

In [ ]:
# 6.1 join the sentiment labels on the sentence key, and the share of negative sentences by bank and window under
# each definition, with 95% intervals from resampling utterances.
# The labels are the sentiment group's own files, one per bank, in the data folder (read only). To use other files change
# LABEL_FILES and, if their columns differ, LABEL_DEFS: name used below -> (label column in the file, value that means negative).
LABEL_FILES = [DATA / f"sentiment_classification_output_{b}.csv" for b in BANKS]
LABEL_DEFS = {"neg_finbert": ("main_sentiment", "negative"),
              "neg_finbert_tone": ("FBT_Main_Sentiment", "Negative"),
              "neg_roberta": ("RBT_main_sentiment", "RBT_negative")}
missing_files = [p.name for p in LABEL_FILES if not p.exists()]
if missing_files:
    raise FileNotFoundError(f"label files {missing_files} are not in {DATA}: copy the sentiment group's output files there "
                            f"or set LABEL_FILES")
USE = KEY + ["section"] + [col for col, _ in LABEL_DEFS.values()]          # text columns are never read
for p in LABEL_FILES:
    absent = [c for c in USE if c not in pd.read_csv(p, nrows=0).columns]
    if absent:
        raise KeyError(f"{p.name} lacks the columns {absent}")
lab = pd.concat([pd.read_csv(p, usecols=USE) for p in LABEL_FILES], ignore_index=True)
lab["key"] = lab[KEY].astype(str).agg("|".join, axis=1)
assert lab["key"].is_unique, "the label files have more than one row for the same sentence key"
neg = pd.DataFrame({"key": lab["key"]})
for name, (col, value) in LABEL_DEFS.items():
    neg[name] = lab[col].astype(str).str.lower().eq(value.lower())
neg["neg_2of3"] = neg[list(LABEL_DEFS)].sum(axis=1) >= 2
qa = qa.drop(columns=[c for c in neg.columns if c != "key" and c in qa.columns])
qa = qa.merge(neg, on="key", how="left", validate="one_to_one")
unlabelled = qa["neg_finbert"].isna()
if unlabelled.any():
    raise ValueError(f"{int(unlabelled.sum()):,} of {len(qa):,} topic-corpus sentences have no label in the label files "
                     f"({len(lab):,} rows, sections {sorted(lab['section'].unique())}); "
                     f"first missing keys: {qa.loc[unlabelled, 'key'].head(3).tolist()}")
NEG = {"FinBERT (main)": "neg_finbert", "at least 2 of 3": "neg_2of3",
       "FinBERT-tone": "neg_finbert_tone", "RoBERTa": "neg_roberta"}

rng = np.random.default_rng(SEED)
rows = []
for (b, w), g in qa.groupby(["bank", "window"]):
    for name, col in NEG.items():
        point, boot = shares(g, col, [False, True], rng)
        lo, hi = np.percentile(boot[:, 1], [2.5, 97.5])
        rows.append({"bank": b, "window": w, "definition": name, "negative %": 100 * point[1],
                     "range": f"{100 * lo:.1f} to {100 * hi:.1f}", "negative sentences": int(g[col].sum())})
neg_share = pd.DataFrame(rows)
neg_share.to_csv(OUT / "negative_share_bank_window.csv", index=False)
print(f"labels from {len(LABEL_FILES)} files ({len(lab):,} sentences, {', '.join(p.name for p in LABEL_FILES)}) "
      f"joined for all {len(qa):,} sentences of the topic corpus")
display(neg_share.pivot_table(index="definition", columns=["bank", "window"], values="negative %")
        .reindex(index=list(NEG), columns=pd.MultiIndex.from_product([BANKS, WINDOW_NAMES])).round(1))
display(neg_share[neg_share["definition"] == "FinBERT (main)"].set_index(["bank", "window"])
        [["negative %", "range", "negative sentences"]].reindex(pd.MultiIndex.from_product([BANKS, WINDOW_NAMES])))

Joins the sentiment group's labels (two files, one per bank, 16,513 sentences) to the 10,059 sentences of the topic corpus; every sentence finds its label. FinBERT calls 5% to 7% of UBS's Q&A sentences negative and 7% to 9% of JPMorgan's, and in each bank the ranges for the year before the deals and the year after overlap, so the share did not clearly change. The definitions differ in level (FinBERT-tone finds the most negative sentences, RoBERTa the fewest at UBS), so comparisons stay within one definition, and each window holds only 29 to 175 negative sentences.

In [ ]:
# 6.2 which topics the negative sentences talk about, before (2022) and after (2023-Q3 to 2024-Q2), per bank:
# the topic mix of FinBERT-negative sentences with 95% intervals from resampling utterances, and the change.
# The strict definition (2 of 3 models) is the check.
def negative_mix(col, rng, df=None):
    df = qa if df is None else df
    rows = []
    for b in BANKS:
        point, boot = {}, {}
        for w in ("before", "after"):
            g = df[(df["bank"] == b) & (df["window"] == w) & df[col]]
            point[w], boot[w] = shares(g, "topic", TOPICS, rng)
        lo, hi = np.percentile(100 * (boot["after"] - boot["before"]), [2.5, 97.5], axis=0)
        rows += [{"bank": b, "topic": t, "before": 100 * point["before"][k], "after": 100 * point["after"][k],
                  "lo": lo[k], "hi": hi[k]} for k, t in enumerate(TOPICS)]
    mix = pd.DataFrame(rows).set_index(["bank", "topic"])
    mix["change"] = mix["after"] - mix["before"]
    mix["clear"] = (mix["lo"] > 0) | (mix["hi"] < 0)
    return mix[["before", "after", "change", "lo", "hi", "clear"]]

rng = np.random.default_rng(SEED)
neg_mix = negative_mix("neg_finbert", rng)
strict_mix = negative_mix("neg_2of3", rng)
neg_mix.insert(0, "name", [topic_label(t) for _, t in neg_mix.index])
neg_mix.round(2).to_csv(OUT / "negative_topics_before_after.csv")
print("same topic mix under the strict definition? correlation of the shares:",
      {f"{b} {w}": round(neg_mix.loc[b, w].corr(strict_mix.loc[b, w]), 2) for b in BANKS for w in ("before", "after")})

fig, axes = plt.subplots(1, 2, figsize=(13, 6.4), sharex=True)
for ax, b in zip(axes, BANKS):
    m = neg_mix.loc[b]
    m = m.loc[m[["before", "after"]].max(axis=1).sort_values(ascending=False).index[:14]]
    y = np.arange(len(m))
    ax.hlines(y, m["before"], m["after"], color="#c3c2b7", lw=2, zorder=1)
    ax.scatter(m["before"], y, s=46, facecolors="white", edgecolors="#898781", linewidths=1.5, zorder=2,
               label="before (2022)")
    ax.scatter(m["after"], y, s=46, color=BANK_COLOR[b], zorder=3, label="after (2023-Q3 to 2024-Q2)")
    ax.set_yticks(y, [topic_label(t) + (" *" if m.at[t, "clear"] else "") for t in m.index], fontsize=8)
    ax.invert_yaxis()
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("% of the bank's FinBERT-negative Q&A sentences")
    ax.set_title(b, loc="left")
    ax.legend(loc="lower right", fontsize=8)
fig.suptitle("Where the negative sentences sit, the year before the deals against the year after",
             x=0.01, ha="left", fontweight="bold")
fig.text(0.01, 0.005, "The 14 topics with the most negative sentences per bank. * the change has a 95% interval "
         "that excludes zero (resampling answers). Negative = FinBERT label from the sentiment track's cleaned text.",
         fontsize=8, color="#52514e")
plt.tight_layout(rect=(0, 0.03, 1, 0.95))
fig.savefig(FIG / "fig_6.2_negative_topics.png", dpi=150, bbox_inches="tight")
plt.show()

print("changes in the topic mix of negative sentences whose 95% interval excludes zero")
clear = neg_mix[neg_mix["clear"]]
display(clear.reindex(clear["change"].abs().sort_values(ascending=False).index).round(1))

Shows which topics the negative sentences are about, in 2022 and in the year after the deal. At UBS, costs goes from none of the negative sentences to 19% (11 of 58), while the outlook topic (7% to none) and wealth management (5% to none) drop out; at JPMorgan, NII and income guidance rises from 6% to 16% and quarterly results fall from 8% to 1%. The strict definition (two of three models) gives a similar topic mix (correlation 0.81 to 0.99).

In [ ]:
# 6.3 which topics are most often negative (share of the topic's sentences that FinBERT calls negative, 2022 to
# 2025, topics with at least 40 sentences at the bank), and the theme mix of negative sentences before and after
rate = (qa.groupby(["bank", "topic"]).agg(sentences=("key", "size"), negative=("neg_finbert", "sum"))
        .query("sentences >= 40"))
rate["negative %"] = 100 * rate["negative"] / rate["sentences"]
rate.insert(0, "name", [topic_label(t) for _, t in rate.index])
rate.round(1).to_csv(OUT / "negative_rate_by_topic.csv")
for b in BANKS:
    print(f"{b}: the most and least negative topics (corpus rate {100 * qa.loc[qa['bank'] == b, 'neg_finbert'].mean():.1f}%)")
    r = rate.loc[b].sort_values("negative %", ascending=False)
    display(pd.concat([r.head(6), r.tail(3)]).round(1))

print("theme mix of FinBERT-negative sentences (% of the bank's negative sentences in the window)")
nq = qa[qa["neg_finbert"] & qa["window"].isin(["before", "after"])]
display((100 * pd.crosstab(nq["theme"], [nq["bank"], nq["window"]], normalize="columns"))
        .reindex(index=THEMES, columns=pd.MultiIndex.from_product([BANKS, ["before", "after"]])).round(1))

Shows which topics are most often negative over 2022 to 2025. At JPMorgan, NII and income (22% of the topic's sentences negative), quarterly results (18%) and deposits (14%) stand out against 7.7% for all its sentences; at UBS quarterly results (15%), lending (12%) and Credit Suisse and Switzerland (9%) stand out against 5.9%, while spoken framing is almost never negative (2%). By theme, capital takes a larger part of UBS's negative sentences after the deal (11% to 26%) and risk a smaller one (21% to 12%); none of UBS's negative sentences in these two windows is tagged integration.

In [ ]:
# 6.4 do the results depend on the courtesy and procedure sentences? The main results keep every sentence of the topic
# corpus, the same sentences the sentiment track labels. Here the sentences tagged in 1.4 and the topics whose main
# theme is "other" (conversation and procedure; rule fixed here, theme from 3.4) are left out, and three results are
# computed again: the before-and-after changes in the topic mix (4.5), the negative share (6.1) and the topic mix of
# the negative sentences (6.2). A result that keeps its direction and its range does not depend on them.
NONSUB = [t for t in topic_table.index if topic_table.at[t, "main theme"] == "other"]
core = qa[~qa["is_procedural"] & ~qa["topic"].isin(NONSUB)].reset_index(drop=True)
print(f"left out: {int(qa['is_procedural'].sum()):,} tagged sentences and {len(NONSUB)} topics "
      f"{[topic_label(t) for t in NONSUB]}; {len(core):,} of {len(qa):,} sentences remain")

moves_core = before_after(core, "topic", TOPICS, np.random.default_rng(SEED))
chk = moves[~moves["topic"].isin(NONSUB + [-1])].merge(moves_core, on=["bank", "topic"], suffixes=("", ", without"))
chk["same direction"] = np.sign(chk["change"]) == np.sign(chk["change, without"])
kept = chk[chk["clear"]]
print(f"(a) topic changes beyond their range in 4.5: {len(kept)}; without the left-out sentences they keep their "
      f"direction in {int(kept['same direction'].sum())} and stay beyond their range in "
      f"{int((kept['same direction'] & kept['clear, without']).sum())}")
chk.round(2).to_csv(OUT / "sensitivity_topic_changes.csv", index=False)

share_all = 100 * qa.groupby(["bank", "window"])["neg_finbert"].mean()
share_core = 100 * core.groupby(["bank", "window"])["neg_finbert"].mean()
print("(b) FinBERT-negative share, % of the bank's sentences in the window")
display(pd.DataFrame({"all sentences": share_all, "without": share_core, "difference": share_core - share_all})
        .reindex(pd.MultiIndex.from_product([BANKS, WINDOW_NAMES])).round(1))

mix_core = negative_mix("neg_finbert", np.random.default_rng(SEED), core)
idx = [i for i in neg_mix.index if i[1] not in NONSUB + [-1]]
a_, c_ = neg_mix.loc[idx], mix_core.loc[idx]
shown = a_["clear"]
same = np.sign(a_.loc[shown, "change"]) == np.sign(c_.loc[shown, "change"])
print(f"(c) changes in the topic mix of negative sentences beyond their range in 6.2: {int(shown.sum())}; without the "
      f"left-out sentences they keep their direction in {int(same.sum())} and stay beyond their range in "
      f"{int((same & c_.loc[shown, 'clear']).sum())}")


Leaves out the 293 tagged sentences and the two topics whose main theme is "other" (spoken framing and a topic about years and cycles), which leaves 7,924 of the 10,059 sentences, and repeats three results. All 6 topic changes of 4.5 and all 4 changes in the topic mix of negative sentences keep their direction and stay beyond their range. The negative share rises by 0.3 to 2.0 points in every window because the left-out sentences are rarely negative, so the levels depend on what is counted but the comparisons do not.

### Summary

The sentiment group's labels, one file per bank, were joined to the topics and themes of all 10,059 sentences. Negative sentences are few in the Q&A (FinBERT: UBS 5% to 7%, JPMorgan 7% to 9% per window), and their share did not clearly change from the year before the deals to the year after. What changed is what they are about: at UBS, negative sentences moved to costs (none in 2022, 19% after the deal) and, by theme, towards capital (11% to 26%), and away from the outlook and wealth management; at JPMorgan they moved to NII and income guidance (6% to 16%) and away from quarterly results. Over the whole period JPMorgan's most negative topics are NII and income (22% of its sentences negative) and quarterly results (18%), while at UBS quarterly results (15%), lending (12%) and Credit Suisse and Switzerland (9%) stand out against 5.9% overall.

These results stand on few sentences (58 negative UBS sentences in the year after the deal) and on labels from cleaned text, where a dropped "not" can turn a reassurance into a negative; leaving out courtesy and procedure sentences changes the levels but not the direction of any comparison (6.4). They show where to read rather than a measured shift in tone.

## Analysts against management

In [ ]:
# 7.1 analyst share minus management share per topic, in percentage points, for each bank over 2022 to 2025.
# Positive: analysts raise the topic more than management talks about it. 95% intervals from resampling
# utterances within each role.
rng = np.random.default_rng(SEED)
gap_rows = []
for b in BANKS:
    for w in ["all"] + WINDOW_NAMES:
        g = qa[(qa["bank"] == b) & ((qa["window"] == w) | (w == "all"))]
        (pa, ba), (pm, bm) = (shares(g[g["speaker_role"] == r], "topic", TOPICS, rng) for r in ROLES)
        lo, hi = np.percentile(100 * (ba - bm), [2.5, 97.5], axis=0)
        gap_rows += [{"bank": b, "window": w, "topic": t, "name": topic_label(t), "gap": 100 * (pa[k] - pm[k]),
                      "lo": lo[k], "hi": hi[k]} for k, t in enumerate(TOPICS)]
gap = pd.DataFrame(gap_rows)
gap.round(2).to_csv(OUT / "analyst_management_topic_gap.csv", index=False)

g = gap[gap["window"] == "all"].set_index(["bank", "topic"])
fig, ax = plt.subplots(figsize=(9, 0.32 * len(show) + 1.6))
y = np.arange(len(show))
for i, b in enumerate(BANKS):
    r = g.xs(b, level="bank").reindex(show)
    yy = y + (i - 0.5) * 0.3
    ax.errorbar(r["gap"], yy, xerr=[r["gap"] - r["lo"], r["hi"] - r["gap"]], fmt="o", ms=5,
                color=BANK_COLOR[b], ecolor=BANK_COLOR[b], elinewidth=1.4, capsize=0, label=b)
ax.axvline(0, color="#52514e", lw=1)
ax.set_yticks(y, [topic_label(t) for t in show], fontsize=8)
ax.invert_yaxis()
ax.grid(axis="y", visible=False)
ax.set_xlabel("percentage points:  ← management talks about it more   ·   analysts ask about it more →")
ax.legend(loc="lower right")
ax.set_title("Analyst share minus management share, by topic, 2022 to 2025", loc="left")
plt.tight_layout()
fig.savefig(FIG / "fig_7.1_analyst_gap_topics.png", dpi=150, bbox_inches="tight")
plt.show()

clear = g[(g["lo"] > 0) | (g["hi"] < 0)].drop(index=-1, level="topic", errors="ignore")
print("topics where the gap's 95% interval excludes zero, largest first")
display(clear.reindex(clear["gap"].abs().sort_values(ascending=False).index)[["name", "gap", "lo", "hi"]]
        .head(16).round(1))

Compares what analysts ask about with what management talks about, by topic. At both banks analysts raise NII and income guidance more than management does (by 2 to 3 points), at UBS also capital ratios (RWA and CET1), and at JPMorgan also lending. Management says more spoken framing with no subject (11 points more at JPMorgan, 4 at UBS), and talks more about wealth and clients at JPMorgan and about Credit Suisse and Switzerland at UBS (2 points each) than its analysts ask about them.

### Summary

Analysts at both banks push on NII and income guidance more than management raises it, at UBS also on capital ratios, and at JPMorgan also on lending. Management fills more of its answers with spoken framing, and talks more about wealth and clients at JPMorgan and about Credit Suisse and Switzerland at UBS than analysts ask about them.

The analyst side shows what the market wants explained. A subject analysts keep pressing that management does not take up is worth a closer look.

## Next

- **Negative topics on raw text.** If the sentiment group moves its models to the raw sentence, put its new output files in the data folder under the same names and rerun sections 5 and 6: negations would then reach the models, and the negative topics may change.
- **Emotion.** The labels file also holds the emotion model's labels. They could describe the negative topics further, as a description only, since that model was trained on tweets.
- **One topic model for the report.** Agree with the topics group which BERTopic setting goes into the final notebook, so the report shows one topic model and its tuning record.
- **Prepared remarks.** The tagger and the topic model can also read the prepared remarks, which would line up with the sentiment track's comparison of prepared remarks and Q&A.
- Before any commit: clear all outputs (public repository) and add only this notebook.